# FC Mold G5 - Data Analysis - TATA Ijmulden


**Goal**
- Identify stable casting sequences
- Remove sensor artifacts
- Relate mold level stability to process parameters

**Data**

CC23 has two strands: 23_5 and 23_6.
- dtExpert , data experts logged with a freq of 1Hz, it includes currents of EMBR parts, and casting parameters
- boExpert, BreakOut experts, logged with a freq of 2Hz, it consists of the raw FBG temperature measurements and some casting parameters

**Last updated**: 2025-06-xx


In [2]:
%pip install mpl-scatter-density
%pip install astropy

   ---------------------------------------- 0.0/749.5 kB ? eta -:--:--
   ---------------------------------------- 749.5/749.5 kB 15.7 MB/s  0:00:00

   ---------------------------------------- 0/2 [fast-histogram]
   -------------------- ------------------- 1/2 [mpl-scatter-density]
   -------------------- ------------------- 1/2 [mpl-scatter-density]
   ---------------------------------------- 2/2 [mpl-scatter-density]

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   --------- ------------------------------ 1.6/6.4 MB 11.9 MB/s eta 0:00:01
   --------- ------------------------------ 1.6/6.4 MB 11.9 MB/s eta 0:00:01
   ------------- -------------------------- 2.1/6.4 MB 4.2 MB/s eta 0:00:02
   ------------------ --------------------- 2.9/6.4 MB 3.5 MB/s eta 0:00:01
   -------------------------- ------------- 4.2/6.4 MB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 6.4/6.4 MB 5.

In [3]:
# Databricks notebook: PySpark + DBFS

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import time

from scipy.ndimage import median_filter
from pathlib import Path

import matplotlib.pyplot as plt
import mpl_scatter_density  # adds projection='scatter_density'
from matplotlib.colors import LinearSegmentedColormap
from astropy.visualization import LogStretch
from astropy.visualization.mpl_normalize import ImageNormalize
import plotly.express as px

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType, LongType, IntegerType, DoubleType, FloatType

from astropy.visualization import LogStretch
from astropy.visualization.mpl_normalize import ImageNormalize

norm = ImageNormalize(vmin=0., vmax=1000, stretch=LogStretch())

# --------------------------------------------------------------------
# Helper to recursively list files in DBFS and split boExpert / dtExpert
# --------------------------------------------------------------------
def add_plain_timestamp(df):
    """
    Ensure df has a 'plainTimeStamp' column of TimestampType, built from
    one of: plainTimeStamp, dt_timestamp_utc, TIMESTAMP.
    """
    schema = df.schema
    cols = df.columns

    # --- Case 1: plainTimeStamp already in data ---
    if "plainTimeStamp" in cols:
        dt = schema["plainTimeStamp"].dataType
        if isinstance(dt, StringType):
            df = df.withColumn("plainTimeStamp", F.to_timestamp("plainTimeStamp"))
        # if it's already TimestampType, do nothing
        return df

    # --- Case 2: dt_timestamp_utc exists (CSV case) ---
    if "dt_timestamp_utc" in cols:
        dt = schema["dt_timestamp_utc"].dataType

        if isinstance(dt, StringType):
            df = df.withColumn("plainTimeStamp", F.to_timestamp("dt_timestamp_utc"))

        elif isinstance(dt, TimestampType):
            df = df.withColumn("plainTimeStamp", F.col("dt_timestamp_utc"))

        elif isinstance(dt, (LongType, IntegerType, DoubleType, FloatType)):
            # numeric epoch: seconds vs milliseconds heuristic
            col = F.col("dt_timestamp_utc")
            df = df.withColumn(
                "plainTimeStamp",
                F.from_unixtime(
                    F.when(col > 1e12, col / 1000).otherwise(col)
                ).cast("timestamp")
            )

        return df

    # --- Case 3: TIMESTAMP exists (parquet case you showed) ---
    if "TIMESTAMP" in cols:
        dt = schema["TIMESTAMP"].dataType

        if isinstance(dt, TimestampType):
            df = df.withColumn("plainTimeStamp", F.col("TIMESTAMP"))

        elif isinstance(dt, (LongType, IntegerType, DoubleType, FloatType)):
            col = F.col("TIMESTAMP")
            df = df.withColumn(
                "plainTimeStamp",
                F.from_unixtime(
                    F.when(col > 1e12, col / 1000).otherwise(col)
                ).cast("timestamp")
            )

        return df

    # --- Fallback: no time column found ---
    return df

def get_expert_files(folder_path: str):
    """
    Databricks/DBFS version.

    folder_path: DBFS or mounted path, e.g. "dbfs:/mnt/tata_fcmoldg5/data/1393"
    Returns:
      bo_expert_files, dt_expert_files  (both lists of full DBFS paths)
    """
    bo_expert_files = []
    dt_expert_files = []

    def walk(path: str):
        for fi in dbutils.fs.ls(path):
            # In Databricks, directories usually end with '/'
            if fi.path.endswith('/'):
                walk(fi.path)
            else:
                name = fi.name
                if 'boExpert' in name:
                    bo_expert_files.append(fi.path)
                elif 'dtExpert' in name:
                    dt_expert_files.append(fi.path)

    walk(folder_path)
    return bo_expert_files, dt_expert_files


def load_expert_files(file_paths):
        """
        file_paths: list of DBFS paths (parquet and/or csv)
        Returns a Spark DataFrame with a proper 'plainTimeStamp' column if possible.
        """
        parquet_files = [f for f in file_paths if f.lower().endswith(".parquet")]
        csv_files     = [f for f in file_paths if f.lower().endswith(".csv")]

        if parquet_files:
            df = spark.read.parquet(*parquet_files)
        elif csv_files:
            df = (
                spark.read
                    .option("header", True)
                    .option("inferSchema", True)
                    .csv(csv_files)
            )
        else:
            return None

        df = add_plain_timestamp(df)
        return df

# Functions to convert units

from pyspark.sql import functions as F

def convert_units(df):
    """
    Convert units in the given Spark DataFrame:
      - castingSpeed:  (m/s → m/min) multiply by 60
      - Mold Level:    (m → mm)      multiply by 1000
      - Argon flows:   (m^3/s → L/min?) multiply by 60000

    Only applies conversions to columns that exist.
    """

    conversions = {
        "castingSpeed": 60,
        "Mold Level": 1000,
        "Mold Level Sensor Left": 1000,
        "Mold Level Sensor Right": 1000,
        "Argon Flow SEN": 60000,
        "Argon Flow Stopper": 60000 #,
        #"SENImmersionDepth": 100
    }

    df_conv = df

    for col, factor in conversions.items():
        if col in df.columns:
            df_conv = df_conv.withColumn(col, F.col(col) * factor)

    return df_conv



In [4]:

%fs ls dbfs:/FileStore/TATA_IJmuiden_CC23/data/


UsageError: Line magic function `%fs` not found.


In [5]:
# Example base path in DBFS (adjust to your environment!)
strand_23_6_path = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/strand_6"

bo_files_23_6, dt_files_23_6 = get_expert_files(strand_23_6_path)


NameError: name 'dbutils' is not defined

In [0]:
dt_files_23_6

## Load boExpert dtExpert files

In [0]:
# bo_files_23_6, dt_files_23_6 already built by get_expert_files(...)

df_boExpert_23_6 = load_expert_files(bo_files_23_6)
df_dtExpert_23_6 = load_expert_files(dt_files_23_6)

# Sanity checks
from pyspark.sql import functions as F

df_boExpert_23_6.select(
    F.count("*").alias("bo_total"),
    F.count("plainTimeStamp").alias("bo_plainTimeStamp_not_null")
).show()

df_dtExpert_23_6.select(
    F.count("*").alias("dt_total"),
    F.count("plainTimeStamp").alias("dt_plainTimeStamp_not_null")
).show()

df_boExpert_23_6.select("plainTimeStamp").show(5, False)
df_dtExpert_23_6.select("plainTimeStamp").show(5, False)


In [0]:
df_boExpert_23_6.printSchema()


In [0]:
df_boExpert_23_6.count()

In [0]:
df_boExpert_23_6.display()

## Join using plainTimeStamp

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

key_col = "plainTimeStamp"

# Identify numeric vs non-numeric columns (excluding the key)
numeric_cols = [
    f.name for f in df_boExpert_23_6.schema.fields
    if isinstance(f.dataType, NumericType) and f.name != key_col
]

non_numeric_cols = [
    f.name for f in df_boExpert_23_6.schema.fields
    if not isinstance(f.dataType, NumericType) and f.name != key_col
]

print("Numeric columns to average:", numeric_cols)
print("Non-numeric columns (take first):", non_numeric_cols)

# Build aggregation expressions
agg_exprs = (
    [F.avg(c).alias(c) for c in numeric_cols] +
    [F.first(c, ignorenulls=True).alias(c) for c in non_numeric_cols]
)

# Aggregate bo on plainTimeStamp
df_boExpert_23_6_agg = (
    df_boExpert_23_6
    .groupBy(key_col)
    .agg(*agg_exprs)
)

# Sanity check: should now equal bo_distinct
print("df_boExpert_23_6_agg rows:", df_boExpert_23_6_agg.count())


In [0]:
from pyspark.sql import functions as F

# What columns exist and their types?
print(df_boExpert_23_6.dtypes)

# Non-null counts
df_boExpert_23_6.select(
    F.count("*").alias("bo_total"),
    F.count("plainTimeStamp").alias("plainTimeStamp_not_null"),
    F.count(F.when(F.col("plainTimeStamp").isNull(), 1)).alias("plainTimeStamp_nulls")
).show()

# Peek at *all* potential time columns
time_cols = [c for c, t in df_boExpert_23_6.dtypes if 'time' in c.lower() or 'timestamp' in c.lower()]
print("Time-like columns:", time_cols)
df_boExpert_23_6.select(*time_cols).show(20, truncate=False)


In [0]:
df_boExpert_23_6.count()

In [0]:
df_boExpert_23_6_agg.count()

In [0]:
df_dtExpert_23_6.display()

In [0]:
df_23_6_spark = (
    df_boExpert_23_6_agg.alias("bo")
    .join(df_dtExpert_23_6.alias("dt"), on=key_col, how="inner")
)

print("joined rows:", df_23_6_spark.count())


df_23_6_spark.cache()
df_23_6_spark.count()


In [0]:
print("bo rows:", df_boExpert_23_6.count())
print("dt rows:", df_dtExpert_23_6.count())
print("joined rows:", df_23_6_spark.count())


In [0]:
from pyspark.sql import functions as F

bo_total = df_boExpert_23_6.count()
dt_total = df_dtExpert_23_6.count()

bo_distinct = df_boExpert_23_6.select("plainTimeStamp").distinct().count()
dt_distinct = df_dtExpert_23_6.select("plainTimeStamp").distinct().count()

print("bo_total     :", bo_total)
print("bo_distinct  :", bo_distinct)
print("bo duplicates:", bo_total - bo_distinct)

print("dt_total     :", dt_total)
print("dt_distinct  :", dt_distinct)
print("dt duplicates:", dt_total - dt_distinct)


In [0]:
df_23_6_spark.count()

In [0]:
metadata_path = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv"
df_metadata_spark = spark.read.csv('dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv', header=True, inferSchema=True, sep=";")
display(df_metadata_spark)


In [0]:
from pyspark.sql import functions as F

#metadata_path = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv"
#df_metadata_spark = spark.read.csv('dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv', header=True, inferSchema=True, sep=";")
#display(df_metadata_spark)

df_23_6_spark = df_23_6_spark.withColumn(
    "PlainTimeStamp",
    F.col("PlainTimeStamp").cast("timestamp")
)

df_metadata_spark = df_metadata_spark \
    .withColumn("Datetime start first heat", F.col("Datetime start first heat").cast("timestamp")) \
    .withColumn("Datetime start last heat",  F.col("Datetime start last heat").cast("timestamp"))

join_condition = (
    (df_23_6_spark["PlainTimeStamp"] >= df_metadata_spark["Datetime start first heat"]) &
    (df_23_6_spark["PlainTimeStamp"] <= df_metadata_spark["Datetime start last heat"])
)

df_23_6_spark = (
    df_23_6_spark
    .join(
        df_metadata_spark.select(
            "Datetime start first heat",
            "Datetime start last heat",
            "Quality casting"
        ),
        on=join_condition,
        how="left"
    )
)



In [0]:
display(df_23_6_spark)

In [0]:
from pyspark.sql import functions as F

df_metadata_spark.select(
    "Datetime start first heat",
    "Datetime start last heat",
    F.col("Datetime start first heat").cast("timestamp").alias("first_cast"),
    F.col("Datetime start last heat").cast("timestamp").alias("last_cast")
).show(20, truncate=False)


In [0]:
from pyspark.sql import functions as F

def parse_ts_with_tz(colname):
    # example: "2025-05-01 21:56:21+02:00" -> "2025-05-01 21:56:21"
    return F.to_timestamp(
        F.regexp_replace(F.col(colname), r"\+\d{2}:\d{2}$", ""),
        "yyyy-MM-dd HH:mm:ss"
    )

df_metadata_fixed = (
    df_metadata_spark
    .withColumn("ts_start", parse_ts_with_tz("Datetime start first heat"))
    .withColumn("ts_end",   parse_ts_with_tz("Datetime start last heat"))
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# 1) Convert PlainTimeStamp (UTC) into local time to match metadata
df_23_6_local = df_23_6_spark.withColumn(
    "PlainTimeStamp_local",
    F.from_utc_timestamp(F.col("PlainTimeStamp"), "Europe/Amsterdam")
)

# Drop "Quality casting" if it exists to avoid duplicate columns in the join
if "Quality casting" in df_23_6_local.columns:
    df_23_6_local = df_23_6_local.drop("Quality casting")

# 2) Parse metadata timestamps and FILTER FOR STRAND 6 ONLY
df_metadata_fixed = (
    df_metadata_spark
    .filter(F.col("Strand ID") == 6)  # ← CRITICAL: Filter for Strand 6 only
    .withColumn("ts_start", F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ts_end",   F.to_timestamp("Datetime start last heat",  "yyyy-MM-dd HH:mm:ss"))
    .filter(F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull())
)

print(f"Metadata records for Strand 6: {df_metadata_fixed.count()}")

# 3) Range join condition in SAME timezone reference (local)
join_condition = (
    (df_23_6_local["PlainTimeStamp_local"] >= df_metadata_fixed["ts_start"]) &
    (df_23_6_local["PlainTimeStamp_local"] <= df_metadata_fixed["ts_end"])
)

# 4) Join (broadcast metadata if small)
df_23_6_joined = (
    df_23_6_local
    .join(
        broadcast(df_metadata_fixed.select("ts_start", "ts_end", "Quality casting", "Casting ID")),
        on=join_condition,
        how="left"
    )
)

# 5) (Optional) drop helper cols
df_23_6_joined = df_23_6_joined.drop("ts_start", "ts_end")

print(f"\nJoin results:")
df_23_6_joined.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("Quality casting").isNotNull().cast("int")).alias("matched_rows"),
    F.countDistinct("Casting ID").alias("unique_castings")
).show()

In [0]:
df_23_6_joined.select(
    F.count("*").alias("rows"),
    F.sum(F.col("Quality casting").isNotNull().cast("int")).alias("matched_rows")
).show()

df_23_6_joined.select("PlainTimeStamp", "PlainTimeStamp_local", "Quality casting") \
    .where(F.col("Quality casting").isNotNull()) \
    .show(20, truncate=False)


In [0]:
%python
display(
    df_23_6_spark.groupBy(df_23_6_spark["Quality casting"]).count()
)

In [0]:
from pyspark.sql import functions as F

# Get statistics on null vs non-null Quality casting
quality_stats = df_23_6_spark.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("Quality casting").isNull(), 1).otherwise(0)).alias("null_quality"),
    F.sum(F.when(F.col("Quality casting").isNotNull(), 1).otherwise(0)).alias("non_null_quality")
).collect()[0]

print("="*70)
print("QUALITY CASTING DISTRIBUTION ANALYSIS")
print("="*70)
print(f"Total rows: {quality_stats['total_rows']:,}")
print(f"Null Quality casting: {quality_stats['null_quality']:,} ({100*quality_stats['null_quality']/quality_stats['total_rows']:.2f}%)")
print(f"Non-null Quality casting: {quality_stats['non_null_quality']:,} ({100*quality_stats['non_null_quality']/quality_stats['total_rows']:.2f}%)")
print("="*70)

# Get time range for null vs non-null
print("\nTIME RANGE COMPARISON:")
print("-"*70)

# Time range for NULL quality casting
null_time_range = df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    F.min("PlainTimeStamp").alias("first_null"),
    F.max("PlainTimeStamp").alias("last_null")
).collect()[0]

print(f"\nNull Quality casting time range:")
print(f"  First: {null_time_range['first_null']}")
print(f"  Last: {null_time_range['last_null']}")

# Time range for NON-NULL quality casting
non_null_time_range = df_23_6_spark.filter(F.col("Quality casting").isNotNull()).select(
    F.min("PlainTimeStamp").alias("first_non_null"),
    F.max("PlainTimeStamp").alias("last_non_null")
).collect()[0]

print(f"\nNon-null Quality casting time range:")
print(f"  First: {non_null_time_range['first_non_null']}")
print(f"  Last: {non_null_time_range['last_non_null']}")
print("="*70)

In [0]:
from pyspark.sql import functions as F

# Get metadata casting period ranges
metadata_range = df_metadata_spark.select(
    F.min(F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss")).alias("first_casting_start"),
    F.max(F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss")).alias("last_casting_end")
).collect()[0]

print("="*70)
print("METADATA CASTING PERIODS")
print("="*70)
print(f"First casting starts: {metadata_range['first_casting_start']}")
print(f"Last casting ends: {metadata_range['last_casting_end']}")
print("="*70)

# Get overall data time range
data_range = df_23_6_spark.select(
    F.min("PlainTimeStamp").alias("data_start"),
    F.max("PlainTimeStamp").alias("data_end")
).collect()[0]

print("\nOVERALL DATA TIME RANGE")
print("="*70)
print(f"Data starts: {data_range['data_start']}")
print(f"Data ends: {data_range['data_end']}")
print("="*70)

# Check if data extends beyond casting periods
print("\nANALYSIS:")
print("-"*70)
if data_range['data_start'] < metadata_range['first_casting_start']:
    print(f"⚠️  Data starts BEFORE first casting period")
    print(f"   Gap: {(metadata_range['first_casting_start'] - data_range['data_start']).total_seconds() / 3600:.2f} hours")
else:
    print(f"✓ Data starts within or after casting periods")

if data_range['data_end'] > metadata_range['last_casting_end']:
    print(f"⚠️  Data ends AFTER last casting period")
    print(f"   Gap: {(data_range['data_end'] - metadata_range['last_casting_end']).total_seconds() / 3600:.2f} hours")
else:
    print(f"✓ Data ends within or before casting periods")
print("="*70)

In [0]:
from pyspark.sql import functions as F, Window

# Get casting periods sorted by time
df_casting_periods_sorted = df_metadata_spark.select(
    F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss").alias("ts_start"),
    F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss").alias("ts_end"),
    "Casting ID",
    "Strand ID"
).filter(
    F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull()
).orderBy("ts_start")

# Calculate gaps between consecutive casting periods
window_spec = Window.orderBy("ts_start")

df_casting_gaps = df_casting_periods_sorted.withColumn(
    "prev_end",
    F.lag("ts_end").over(window_spec)
).withColumn(
    "gap_hours",
    (F.unix_timestamp("ts_start") - F.unix_timestamp("prev_end")) / 3600
).filter(
    F.col("gap_hours").isNotNull() & (F.col("gap_hours") > 0)
)

print("="*70)
print("GAPS BETWEEN CASTING PERIODS")
print("="*70)
print(f"Number of gaps between castings: {df_casting_gaps.count()}")
print("\nTop 10 largest gaps:")
df_casting_gaps.select(
    "prev_end",
    "ts_start",
    "gap_hours",
    "Casting ID",
    "Strand ID"
).orderBy(F.desc("gap_hours")).show(10, truncate=False)

# Calculate total gap time
total_gap_hours = df_casting_gaps.select(F.sum("gap_hours")).collect()[0][0]
print(f"\nTotal gap time between castings: {total_gap_hours:.2f} hours")
print("="*70)

In [0]:
from pyspark.sql import functions as F

# Sample some null Quality casting records to see their timestamps
print("="*70)
print("SAMPLE RECORDS WITH NULL QUALITY CASTING")
print("="*70)
print("\nFirst 20 records with null Quality casting:")
df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    "PlainTimeStamp",
    "Quality casting"
).orderBy("PlainTimeStamp").show(20, truncate=False)

print("\nLast 20 records with null Quality casting:")
df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    "PlainTimeStamp",
    "Quality casting"
).orderBy(F.desc("PlainTimeStamp")).show(20, truncate=False)
print("="*70)

## Analysis Summary: Null Quality Casting Values

### Key Findings:

**1. Distribution:**
* **45.74%** of measurements (324,079 rows) have **NULL** Quality casting
* **54.26%** of measurements (384,376 rows) have **non-null** Quality casting

**2. Temporal Analysis:**

**Null Quality Casting Time Range:**
* First: `2025-04-14 01:09:46`
* Last: `2025-05-16 05:36:25`
* Spans throughout the entire dataset

**Non-null Quality Casting Time Range:**
* First: `2025-04-13 22:59:26`
* Last: `2025-04-30 10:09:43`
* Limited to specific casting periods

**3. Root Cause:**

⚠️ **YES - Null values are from measurements OUTSIDE casting periods:**

* **Metadata casting periods:** April 13, 21:14:33 → May 5, 05:53:52
* **Actual data range:** April 13, 22:59:26 → **May 16, 05:36:25**
* **Data extends 263.71 hours (11 days) AFTER the last casting period**

**4. Explanation:**

The null `Quality casting` values occur because:

1. **Between casting periods:** 26 gaps totaling 410.73 hours between consecutive castings
   - Largest gap: 78.6 hours (April 18 → April 21)
   - During these gaps, the MEX system continues recording data but no casting is active

2. **After last casting:** 263.71 hours of data after May 5, 05:53:52
   - System continues logging even though no casting metadata exists
   - This is the majority of null values

3. **Before first casting:** Minimal (data starts within casting periods)

### Recommendation:

For your **mold level fluctuation analysis**, you should:
* **Filter out null Quality casting rows** - they represent non-production periods
* **Focus on sequences with valid Quality casting** - these are actual casting operations
* **Use Quality casting as a grouping variable** for sequence-based analysis

In [0]:
import plotly.graph_objects as go
import pandas as pd
from pyspark.sql import functions as F

# Sample data for visualization (to avoid memory issues)
sample_rate = max(1, df_23_6_spark.count() // 10000)

df_sample = df_23_6_spark.sample(fraction=1.0/sample_rate, seed=42).select(
    "PlainTimeStamp",
    F.when(F.col("Quality casting").isNull(), "No Casting (NULL)").otherwise("Active Casting").alias("status"),
    "Quality casting"
).orderBy("PlainTimeStamp").toPandas()

# Create figure
fig = go.Figure()

# Add traces for null and non-null
for status in df_sample['status'].unique():
    df_status = df_sample[df_sample['status'] == status]
    color = 'red' if status == 'No Casting (NULL)' else 'green'
    
    fig.add_trace(go.Scatter(
        x=df_status['PlainTimeStamp'],
        y=[1] * len(df_status),
        mode='markers',
        marker=dict(color=color, size=3, opacity=0.5),
        name=status,
        hovertemplate='%{x}<br>Status: ' + status + '<extra></extra>'
    ))

# Add metadata casting period boundaries
metadata_periods = df_metadata_spark.select(
    F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss").alias("start"),
    F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss").alias("end"),
    "Casting ID"
).filter(F.col("start").isNotNull()).toPandas()

for idx, row in metadata_periods.iterrows():
    fig.add_vrect(
        x0=row['start'], x1=row['end'],
        fillcolor='lightblue', opacity=0.1,
        layer="below", line_width=0
    )

fig.update_layout(
    title=dict(
        text='Temporal Distribution: Null vs Non-Null Quality Casting',
        font=dict(size=16, color='darkblue')
    ),
    xaxis_title='Timeline',
    yaxis_title='Data Presence',
    height=500,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.15,
        xanchor="center",
        x=0.5
    ),
    yaxis=dict(showticklabels=False, range=[0.5, 1.5]),
    hovermode='closest'
)

fig.show()

print("\n" + "="*70)
print("VISUALIZATION EXPLANATION")
print("="*70)
print("• GREEN dots: Measurements during active casting (with Quality casting)")
print("• RED dots: Measurements outside casting periods (NULL Quality casting)")
print("• Light blue bands: Metadata-defined casting periods")
print("\nNotice: Red dots appear in gaps between blue bands and after the last band")
print("="*70)

## Features engineering

In [0]:

df_23_6_spark_unitConverted = convert_units(df_23_6_spark)
#df_23_6_spark_unitConverted = convert_argon_flow_to_l_per_min(df_23_6_spark_unitConverted)
display(df_23_6_spark_unitConverted)

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Sort by timestamp and calculate time difference between consecutive rows
window_spec = Window.orderBy("PlainTimeStamp")

df_time_gaps = (
    df_23_6_spark_unitConverted
    .select("PlainTimeStamp")
    .withColumn("prev_timestamp", F.lag("PlainTimeStamp").over(window_spec))
    .withColumn(
        "time_diff_seconds",
        F.unix_timestamp("PlainTimeStamp") - F.unix_timestamp("prev_timestamp")
    )
    .filter(F.col("time_diff_seconds").isNotNull())
)

# Show statistics about time gaps
df_time_gaps.select(
    F.min("time_diff_seconds").alias("min_gap_sec"),
    F.max("time_diff_seconds").alias("max_gap_sec"),
    F.avg("time_diff_seconds").alias("avg_gap_sec"),
    F.expr("percentile(time_diff_seconds, 0.5)").alias("median_gap_sec"),
    F.expr("percentile(time_diff_seconds, 0.95)").alias("p95_gap_sec")
).show()

In [0]:
# Define threshold for "significant" gap (e.g., > 5 minutes = 300 seconds)
threshold_seconds = 300

df_significant_gaps = (
    df_time_gaps
    .filter(F.col("time_diff_seconds") > threshold_seconds)
    .withColumn("time_diff_minutes", F.col("time_diff_seconds") / 60)
    .withColumn("time_diff_hours", F.col("time_diff_seconds") / 3600)
    .orderBy(F.desc("time_diff_seconds"))
)

print(f"Found {df_significant_gaps.count()} gaps larger than {threshold_seconds} seconds ({threshold_seconds/60} minutes)")
display(df_significant_gaps.select(
    "prev_timestamp",
    "PlainTimeStamp",
    "time_diff_seconds",
    "time_diff_minutes",
    "time_diff_hours"
).limit(20))

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Convert to pandas for plotting
df_gaps_pd = df_time_gaps.select("PlainTimeStamp", "time_diff_seconds").toPandas()

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Time Series Gap Analysis', fontsize=16, fontweight='bold')

# 1. Histogram of time gaps (log scale)
ax1 = axes[0, 0]
ax1.hist(df_gaps_pd['time_diff_seconds'], bins=100, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Time Gap (seconds)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Time Gaps')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# 2. Time gaps over time
ax2 = axes[0, 1]
ax2.scatter(df_gaps_pd['PlainTimeStamp'], df_gaps_pd['time_diff_seconds'], 
            alpha=0.5, s=1)
ax2.set_xlabel('Timestamp')
ax2.set_ylabel('Time Gap (seconds)')
ax2.set_title('Time Gaps Over Time')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# 3. Cumulative distribution of gaps
ax3 = axes[1, 0]
sorted_gaps = np.sort(df_gaps_pd['time_diff_seconds'])
cumulative = np.arange(1, len(sorted_gaps) + 1) / len(sorted_gaps) * 100
ax3.plot(sorted_gaps, cumulative, linewidth=2)
ax3.set_xlabel('Time Gap (seconds)')
ax3.set_ylabel('Cumulative Percentage (%)')
ax3.set_title('Cumulative Distribution of Time Gaps')
ax3.set_xscale('log')
ax3.grid(True, alpha=0.3)
ax3.axhline(y=95, color='r', linestyle='--', label='95th percentile')
ax3.legend()

# 4. Box plot of time gaps (filtered to show detail)
ax4 = axes[1, 1]
# Filter out extreme outliers for better visualization
q99 = df_gaps_pd['time_diff_seconds'].quantile(0.99)
filtered_gaps = df_gaps_pd[df_gaps_pd['time_diff_seconds'] <= q99]['time_diff_seconds']
ax4.boxplot(filtered_gaps, vert=True)
ax4.set_ylabel('Time Gap (seconds)')
ax4.set_title(f'Box Plot of Time Gaps (up to 99th percentile: {q99:.1f}s)')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/time_gaps_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal records analyzed: {len(df_gaps_pd):,}")
print(f"Mean gap: {df_gaps_pd['time_diff_seconds'].mean():.2f} seconds")
print(f"Median gap: {df_gaps_pd['time_diff_seconds'].median():.2f} seconds")
print(f"Max gap: {df_gaps_pd['time_diff_seconds'].max():.2f} seconds ({df_gaps_pd['time_diff_seconds'].max()/3600:.2f} hours)")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# Get significant gaps for visualization
df_sig_gaps_pd = df_significant_gaps.select(
    "prev_timestamp", 
    "PlainTimeStamp", 
    "time_diff_hours"
).toPandas()

# Create timeline visualization
fig, ax = plt.subplots(figsize=(16, 6))

# Plot the continuous timeline
if len(df_gaps_pd) > 0:
    # Sample data points for timeline (every 100th point to avoid overcrowding)
    sample_indices = range(0, len(df_gaps_pd), max(1, len(df_gaps_pd) // 1000))
    timeline_sample = df_gaps_pd.iloc[sample_indices]
    
    ax.scatter(timeline_sample['PlainTimeStamp'], 
               [1] * len(timeline_sample), 
               alpha=0.3, s=10, c='blue', label='Data points')

# Mark significant gaps
if len(df_sig_gaps_pd) > 0:
    for idx, row in df_sig_gaps_pd.iterrows():
        ax.axvspan(row['prev_timestamp'], row['PlainTimeStamp'], 
                   alpha=0.3, color='red')
        # Add text label for large gaps
        if row['time_diff_hours'] > 1:
            mid_time = row['prev_timestamp'] + (row['PlainTimeStamp'] - row['prev_timestamp']) / 2
            ax.text(mid_time, 1.1, f"{row['time_diff_hours']:.1f}h", 
                   rotation=90, ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Timestamp', fontsize=12)
ax.set_ylabel('Data Presence', fontsize=12)
ax.set_title(f'Time Series Timeline with Gaps > {threshold_seconds/60} minutes (Red Regions)', 
             fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.5)
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45, ha='right')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', alpha=0.3, label='Data points'),
    Patch(facecolor='red', alpha=0.3, label=f'Gaps > {threshold_seconds/60} min')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
fig.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/timeline_with_gaps.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVisualization shows {len(df_sig_gaps_pd)} significant gaps in the timeline")

In [0]:
from pyspark.sql import functions as F

# Parse metadata timestamps and filter valid records
df_casting_periods = (
    df_metadata_spark
    .withColumn("ts_start", F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ts_end", F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss"))
    .filter(F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull())
    .select(
        "Casting ID",
        "Strand ID",
        "ts_start",
        "ts_end",
        "Quality casting",
        "Heat Count"
    )
    .orderBy("ts_start")
)

print(f"Total casting periods: {df_casting_periods.count()}")
display(df_casting_periods.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Join gaps with casting periods to classify them
df_gaps_classified = df_significant_gaps.alias("gaps").join(
    broadcast(df_casting_periods.alias("cast")),
    (
        (F.col("gaps.prev_timestamp") >= F.col("cast.ts_start")) &
        (F.col("gaps.prev_timestamp") <= F.col("cast.ts_end")) &
        (F.col("gaps.PlainTimeStamp") >= F.col("cast.ts_start")) &
        (F.col("gaps.PlainTimeStamp") <= F.col("cast.ts_end"))
    ),
    how="left"
).select(
    F.col("gaps.prev_timestamp"),
    F.col("gaps.PlainTimeStamp"),
    F.col("gaps.time_diff_hours"),
    F.col("cast.Casting ID").alias("casting_id"),
    F.col("cast.Strand ID").alias("strand_id"),
    F.col("cast.Quality casting").alias("quality"),
    F.when(F.col("cast.Casting ID").isNotNull(), "Within Casting")
     .otherwise("Between Castings").alias("gap_type")
)

print("\nGap Classification:")
df_gaps_classified.groupBy("gap_type").agg(
    F.count("*").alias("count"),
    F.sum("time_diff_hours").alias("total_hours"),
    F.avg("time_diff_hours").alias("avg_hours")
).show()

print("\nGaps WITHIN casting periods (potential issues):")
display(
    df_gaps_classified
    .filter(F.col("gap_type") == "Within Casting")
    .orderBy(F.desc("time_diff_hours"))
)

In [0]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from datetime import datetime
import os

# Convert casting periods to pandas
df_casting_pd = df_casting_periods.toPandas()
df_gaps_class_pd = df_gaps_classified.toPandas()

# Filter for Strand 6 only (Strand 5 data not loaded yet)
df_casting_pd = df_casting_pd[df_casting_pd['Strand ID'] == 5].reset_index(drop=True)

# ---- helpers to restrict events to Strand-6 casting windows ----
# Build IntervalIndex of Strand-6 casting periods
casting_intervals = pd.IntervalIndex.from_arrays(
    df_casting_pd["ts_start"],
    df_casting_pd["ts_end"],
    closed="both"
)

def in_any_casting_window(ts_series):
    """Return boolean mask: timestamp is inside any Strand-6 casting interval."""
    # get_indexer gives -1 when not contained in any interval
    return casting_intervals.get_indexer(ts_series) >= 0

def gap_overlaps_any_casting(row):
    """True if [prev_timestamp, PlainTimeStamp] overlaps ANY Strand-6 casting window."""
    a = row["prev_timestamp"]
    b = row["PlainTimeStamp"]
    if pd.isna(a) or pd.isna(b):
        return False
    # overlap test with all intervals: overlap exists if max(starts) < min(ends)
    # here do it simply: does either end fall inside, or does interval cover a casting boundary
    if in_any_casting_window(pd.Series([a]))[0] or in_any_casting_window(pd.Series([b]))[0]:
        return True
    # also catch gaps spanning across a casting window without endpoints inside
    return ((df_casting_pd["ts_start"] <= b) & (df_casting_pd["ts_end"] >= a)).any()




# Ensure timestamp columns are datetime objects (not Timedelta)
for col in ['ts_start', 'ts_end']:
    if col in df_casting_pd.columns:
        df_casting_pd[col] = pd.to_datetime(df_casting_pd[col])

for col in ['prev_timestamp', 'PlainTimeStamp']:
    if col in df_gaps_class_pd.columns:
        df_gaps_class_pd[col] = pd.to_datetime(df_gaps_class_pd[col])

if 'PlainTimeStamp' in df_gaps_pd.columns:
    df_gaps_pd['PlainTimeStamp'] = pd.to_datetime(df_gaps_pd['PlainTimeStamp'])

# Create output directory if it doesn't exist
output_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/'
os.makedirs(output_dir, exist_ok=True)

# ========================================================================
# FIGURE 1: Casting Periods as Horizontal Timeline (Gantt-style)
# ========================================================================

# Prepare data for Gantt chart using plotly express
df_gantt = df_casting_pd.copy()
df_gantt['Casting_Label'] = df_gantt['Casting ID'].apply(lambda x: f"Casting {int(x)}")
df_gantt['Duration_hours'] = (df_gantt['ts_end'] - df_gantt['ts_start']).dt.total_seconds() / 3600

# Create Gantt chart using plotly express timeline
fig1 = px.timeline(
    df_gantt,
    x_start='ts_start',
    x_end='ts_end',
    y='Casting_Label',
    color_discrete_sequence=['steelblue'],
    title='Casting Periods Timeline - Strand 6',
    labels={'Casting_Label': 'Casting ID'},
    hover_data={
        'Quality casting': True,
        'Heat Count': True,
        'ts_start': '|%Y-%m-%d %H:%M',
        'ts_end': '|%Y-%m-%d %H:%M',
        'Duration_hours': ':.1f',
        'Casting_Label': False
    }
)

# Add quality labels alternating above and below bars
for idx, row in df_gantt.iterrows():
    # Calculate middle of the bar for x position
    mid_time = row['ts_start'] + (row['ts_end'] - row['ts_start']) / 2
    
    # Alternate position: even indices above, odd indices below
    if idx % 2 == 0:
        # Position above the bar
        y_offset = 25  # pixels above
        arrow_direction = -15  # arrow points down to bar
    else:
        # Position below the bar
        y_offset = -25  # pixels below
        arrow_direction = 15  # arrow points up to bar
    
    fig1.add_annotation(
        x=mid_time,
        y=row['Casting_Label'],
        text=row['Quality casting'],
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.5,
        arrowcolor='gray',
        ax=0,  # No horizontal offset
        ay=arrow_direction,  # Vertical arrow pointing to bar
        font=dict(size=10, color='darkblue', weight='bold'),
        xanchor='center',
        yanchor='middle',
        yshift=y_offset,  # Shift text up or down
        bgcolor='rgba(255, 255, 255, 0.8)',  # White background for readability
        bordercolor='lightgray',
        borderwidth=1,
        borderpad=3
    )

fig1.update_layout(
    xaxis_title='Date and Time',
    yaxis_title='Casting ID',
    height=max(600, len(df_casting_pd) * 25),
    showlegend=False,
    hovermode='closest',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    xaxis=dict(
        showgrid=True,
        gridcolor='white',
        gridwidth=1,
        type='date'
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='white',
        gridwidth=1,
        autorange='reversed'
    ),
    title_font=dict(size=18, color='darkblue'),
    margin=dict(t=100, b=80)  # Add top and bottom margins for labels
)

# Update bar appearance
fig1.update_traces(
    marker=dict(opacity=0.7, line=dict(color='darkblue', width=1))
)

fig1.show()

# Save Figure 1
fig1_path = os.path.join(output_dir, 'casting_periods_timeline_strand6.html')
fig1.write_html(fig1_path)
print(f"\n✓ Figure 1 saved to: {fig1_path}")

print(f"\n{'='*70}")
print(f"FIGURE 1: Casting Periods Overview (Strand 6)")
print(f"{'='*70}")
print(f"Total casting periods: {len(df_casting_pd)}")
print(f"Date range: {df_casting_pd['ts_start'].min().strftime('%Y-%m-%d')} to {df_casting_pd['ts_end'].max().strftime('%Y-%m-%d')}")
print(f"Total casting time: {(df_casting_pd['ts_end'] - df_casting_pd['ts_start']).sum().total_seconds() / 3600:.1f} hours")


# df_gaps_pd must exist; if not, skip
if "PlainTimeStamp" in df_gaps_pd.columns:
    df_gaps_pd = df_gaps_pd.copy()
    df_gaps_pd["PlainTimeStamp"] = pd.to_datetime(df_gaps_pd["PlainTimeStamp"], errors="coerce")

    # keep only points inside Strand-6 casting windows
    df_gaps_pd = df_gaps_pd[in_any_casting_window(df_gaps_pd["PlainTimeStamp"])].reset_index(drop=True)

df_gaps_class_pd = df_gaps_class_pd.copy()

# Ensure datetimes (again safe)
df_gaps_class_pd["prev_timestamp"] = pd.to_datetime(df_gaps_class_pd["prev_timestamp"], errors="coerce")
df_gaps_class_pd["PlainTimeStamp"] = pd.to_datetime(df_gaps_class_pd["PlainTimeStamp"], errors="coerce")

# Keep only gaps that overlap Strand-6 casting windows
df_gaps_class_pd = df_gaps_class_pd[
    df_gaps_class_pd.apply(gap_overlaps_any_casting, axis=1)
].reset_index(drop=True)

# OPTIONAL: recompute gaps BETWEEN Strand-6 castings from df_casting_pd
df_casting_sorted = df_casting_pd.sort_values("ts_start").reset_index(drop=True)

between_rows = []
for i in range(1, len(df_casting_sorted)):
    prev_end = df_casting_sorted.loc[i-1, "ts_end"]
    cur_start = df_casting_sorted.loc[i, "ts_start"]
    if cur_start > prev_end:
        between_rows.append({
            "prev_timestamp": prev_end,
            "PlainTimeStamp": cur_start,
            "time_diff_hours": (cur_start - prev_end).total_seconds() / 3600.0,
            "gap_type": "Between Castings",
            "casting_id": None
        })

df_between_strand6 = pd.DataFrame(between_rows)



# ========================================================================
# FIGURE 2: Data Points and Gaps Timeline
# ========================================================================
fig2 = go.Figure()

# Add casting period background bands (without annotations to avoid timestamp arithmetic issues)
for idx, row in df_casting_pd.iterrows():
    fig2.add_vrect(
        x0=row['ts_start'], x1=row['ts_end'],
        fillcolor='lightblue', opacity=0.25,
        layer="below", line_width=0
    )

# Add gaps between castings (orange) and within castings (red)
if len(df_gaps_class_pd) > 0:
    #gaps_between = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Between Castings']
    gaps_between = df_between_strand6
    gaps_within = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Within Casting']
    
    # Add gap rectangles for gaps between castings
    for idx, row in gaps_between.iterrows():
        fig2.add_trace(
            go.Scatter(
                x=[row['prev_timestamp'], row['PlainTimeStamp'], row['PlainTimeStamp'], row['prev_timestamp'], row['prev_timestamp']],
                y=[0.5, 0.5, 1.5, 1.5, 0.5],
                fill='toself',
                fillcolor='orange',
                opacity=0.4,
                line=dict(width=0),
                mode='lines',
                showlegend=False,
                hovertemplate=f"<b>Gap Between Castings</b><br>" +
                             f"Duration: {row['time_diff_hours']:.1f}h<br>" +
                             f"From: {row['prev_timestamp'].strftime('%Y-%m-%d %H:%M')}<br>" +
                             f"To: {row['PlainTimeStamp'].strftime('%Y-%m-%d %H:%M')}<extra></extra>"
            )
        )
    
    # Add gap rectangles for gaps within castings
    for idx, row in gaps_within.iterrows():
        fig2.add_trace(
            go.Scatter(
                x=[row['prev_timestamp'], row['PlainTimeStamp'], row['PlainTimeStamp'], row['prev_timestamp'], row['prev_timestamp']],
                y=[0.5, 0.5, 1.5, 1.5, 0.5],
                fill='toself',
                fillcolor='red',
                opacity=0.6,
                line=dict(width=0),
                mode='lines',
                showlegend=False,
                hovertemplate=f"<b>⚠️ Gap WITHIN Casting (Data Loss!)</b><br>" +
                             f"Duration: {row['time_diff_hours']:.1f}h<br>" +
                             f"Casting: {int(row['casting_id']) if pd.notna(row['casting_id']) else 'Unknown'}<br>" +
                             f"From: {row['prev_timestamp'].strftime('%Y-%m-%d %H:%M')}<br>" +
                             f"To: {row['PlainTimeStamp'].strftime('%Y-%m-%d %H:%M')}<extra></extra>"
            )
        )

# Add data points (sampled)
if len(df_gaps_pd) > 0:
    sample_indices = range(0, len(df_gaps_pd), max(1, len(df_gaps_pd) // 3000))
    timeline_sample = df_gaps_pd.iloc[sample_indices]
    
    fig2.add_trace(
        go.Scatter(
            x=timeline_sample['PlainTimeStamp'],
            y=[1] * len(timeline_sample),
            mode='markers',
            marker=dict(color='green', size=2, opacity=0.3),
            name='Data points',
            hovertemplate='Data point at: %{x}<extra></extra>'
        )
    )

# Add legend items
fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='lightblue', opacity=0.25, line=dict(color='steelblue', width=1)),
        name='Casting period',
        showlegend=True
    )
)

fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='red', opacity=0.6),
        name='⚠️ Gap WITHIN casting (data loss)',
        showlegend=True
    )
)

fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='orange', opacity=0.4),
        name='Gap BETWEEN castings (expected)',
        showlegend=True
    )
)

fig2.update_layout(
    title=dict(
        text='Data Collection Timeline with Gaps - Strand 6',
        font=dict(size=18, color='darkblue')
    ),
    xaxis_title='Date and Time',
    yaxis_title='Data Presence',
    height=600,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.12,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="black",
        borderwidth=1
    ),
    hovermode='closest',
    yaxis=dict(
        showticklabels=False,
        range=[0.4, 1.6],
        showgrid=False
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=0.5,
        type='date'
    ),
    plot_bgcolor='white'
)

fig2.show()

# Save Figure 2
fig2_path = os.path.join(output_dir, 'data_quality_timeline_strand6.html')
fig2.write_html(fig2_path)
print(f"\n✓ Figure 2 saved to: {fig2_path}")

print(f"\n{'='*70}")
print(f"FIGURE 2: Data Quality Analysis (Strand 6)")
print(f"{'='*70}")
if len(df_gaps_class_pd) > 0:
    gaps_within = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Within Casting']
    gaps_between = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Between Castings']
    print(f"✓ Data points analyzed: {len(df_gaps_pd):,}")
    print(f"⚠️  Gaps WITHIN castings: {len(gaps_within)} (RED - data collection issues)")
    print(f"✓ Gaps BETWEEN castings: {len(gaps_between)} (ORANGE - expected downtime)")
    if len(gaps_within) > 0:
        total_loss_hours = gaps_within['time_diff_hours'].sum()
        print(f"⚠️  Total data loss during casting: {total_loss_hours:.1f} hours")
else:
    print(f"✓ No significant gaps found")
print(f"{'='*70}\n")

print(f"\n{'='*70}")
print(f"SAVED FILES")
print(f"{'='*70}")
print(f"Figure 1: {fig1_path}")
print(f"Figure 2: {fig2_path}")
print(f"\nAccess via: https://<databricks-instance>/files/TATAIjmulden_FCMoldG5/figures/")
print(f"{'='*70}\n")

In [0]:
display(dbutils.fs.ls('dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/'))

In [0]:
from pyspark.sql import functions as F

# Find data points that fall outside any casting period
df_data_with_casting = (
    df_23_6_spark_unitConverted
    .select("PlainTimeStamp")
    .alias("data")
    .join(
        broadcast(df_casting_periods.alias("cast")),
        (
            (F.col("data.PlainTimeStamp") >= F.col("cast.ts_start")) &
            (F.col("data.PlainTimeStamp") <= F.col("cast.ts_end"))
        ),
        how="left"
    )
)

# Count matched vs unmatched
matching_stats = df_data_with_casting.select(
    F.count("*").alias("total_data_points"),
    F.sum(F.when(F.col("Casting ID").isNotNull(), 1).otherwise(0)).alias("matched_to_casting"),
    F.sum(F.when(F.col("Casting ID").isNull(), 1).otherwise(0)).alias("unmatched_orphan_data")
).collect()[0]

print("\n" + "="*60)
print("DATA-METADATA MISMATCH ANALYSIS")
print("="*60)
print(f"\nTotal data points: {matching_stats['total_data_points']:,}")
print(f"Matched to casting periods: {matching_stats['matched_to_casting']:,} ({100*matching_stats['matched_to_casting']/matching_stats['total_data_points']:.2f}%)")
print(f"Orphan data (outside any casting): {matching_stats['unmatched_orphan_data']:,} ({100*matching_stats['unmatched_orphan_data']/matching_stats['total_data_points']:.2f}%)")

# Show time range of orphan data
orphan_data = df_data_with_casting.filter(F.col("Casting ID").isNull())
if orphan_data.count() > 0:
    orphan_range = orphan_data.select(
        F.min("PlainTimeStamp").alias("first_orphan"),
        F.max("PlainTimeStamp").alias("last_orphan")
    ).collect()[0]
    
    print(f"\nOrphan data time range:")
    print(f"  First: {orphan_range['first_orphan']}")
    print(f"  Last: {orphan_range['last_orphan']}")

# Check for casting periods with no data
print("\n" + "-"*60)
print("Casting periods with data coverage:")
print("-"*60)

casting_coverage = (
    df_casting_periods.alias("cast")
    .join(
        df_23_6_spark_unitConverted.select("PlainTimeStamp").alias("data"),
        (
            (F.col("data.PlainTimeStamp") >= F.col("cast.ts_start")) &
            (F.col("data.PlainTimeStamp") <= F.col("cast.ts_end"))
        ),
        how="left"
    )
    .groupBy("Casting ID", "Strand ID", "ts_start", "ts_end", "Quality casting")
    .agg(
        F.count("PlainTimeStamp").alias("data_point_count"),
        F.min("PlainTimeStamp").alias("first_data"),
        F.max("PlainTimeStamp").alias("last_data")
    )
    .withColumn(
        "coverage_status",
        F.when(F.col("data_point_count") == 0, "NO DATA")
         .when(F.col("first_data") > F.col("ts_start"), "LATE START")
         .when(F.col("last_data") < F.col("ts_end"), "EARLY END")
         .otherwise("FULL COVERAGE")
    )
    .orderBy("ts_start")
)

display(casting_coverage)

# Summary by coverage status
print("\nCoverage Summary:")
casting_coverage.groupBy("coverage_status").count().orderBy("coverage_status").show()

In [0]:
# Create detailed mismatch report for further investigation
mismatch_report = casting_coverage.filter(
    F.col("coverage_status") != "FULL COVERAGE"
).select(
    "Casting ID",
    "Strand ID",
    "ts_start",
    "ts_end",
    "Quality casting",
    "data_point_count",
    "first_data",
    "last_data",
    "coverage_status"
).withColumn(
    "expected_duration_hours",
    (F.unix_timestamp("ts_end") - F.unix_timestamp("ts_start")) / 3600
).withColumn(
    "data_start_delay_hours",
    F.when(
        F.col("first_data").isNotNull(),
        (F.unix_timestamp("first_data") - F.unix_timestamp("ts_start")) / 3600
    ).otherwise(None)
).withColumn(
    "data_end_early_hours",
    F.when(
        F.col("last_data").isNotNull(),
        (F.unix_timestamp("ts_end") - F.unix_timestamp("last_data")) / 3600
    ).otherwise(None)
)

print(f"\nCastings with data mismatches: {mismatch_report.count()}")
display(mismatch_report.orderBy("ts_start"))

# Save report
import os
report_path = "/dbfs/FileStore/TATAIjmulden_FCMoldG5/reports/casting_data_mismatch_report.csv"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
mismatch_report.toPandas().to_csv(report_path, index=False)
print(f"\nMismatch report saved to: {report_path}")

In [0]:
df_23_6_spark_clean = (
    df_23_6_spark_unitConverted
        .filter(F.col("castingSpeed") >= 0.5)
        .filter(F.col("SENImmersionDepth").between(0.1, 0.26))
        #.filter(F.col("moldWidth").isNotNull())
)


In [0]:
print("Before:", df_23_6_spark_unitConverted.count())
print("After :", df_23_6_spark_clean.count())


## Visualization: Select small subset for plotting

In [0]:
import os
fig_dir = "dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/"
os.makedirs(fig_dir, exist_ok=True)

In [0]:
cols_for_plot = [
    "PlainTimeStamp",
    "castingSpeed",
    "moldWidth",
    "SENImmersionDepth",
    'Mold Level', 
    'Mold Level Sensor Left', 
    'Mold Level Sensor Right',
    'Argon Flow SEN', 'Argon Flow Stopper',
    'EMBR Current DC Left Bottom',
    'EMBR Current AC Left Master',
    'EMBR Current DC Left Master',
    'Quality casting'
]

df_23_6_small = (
    df_23_6_spark_clean
       .select(*cols_for_plot)
        .dropna(subset=["castingSpeed", "moldWidth", "SENImmersionDepth"])
        # Optional if datasets are huge:
        # .sample(False, 0.1, seed=42)
        # .limit(500000)
)

df_23_6 = df_23_6_small.toPandas()
df_23_6 = df_23_6.rename(columns={"PlainTimeStamp": "plainTimeStamp"})

In [0]:
display(df_23_6_small)

In [0]:
df_23_6_small.printSchema()

In [0]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# --- Create subplots container ---
fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Casting Speed Distribution",
        "SENID Distribution",
        "Mold Width Distribution",
        "Argon Flow SEN+Stopper Distribution"
    ]
)

# --- Subplot 1: Casting Speed ---
h1 = px.histogram(df_23_6, x='castingSpeed')
fig1.add_trace(h1.data[0], row=1, col=1)
fig1.update_xaxes(title_text="Casting Speed [m/min]", row=1, col=1)
fig1.update_yaxes(title_text="Count", row=1, col=1)

# --- Subplot 2: SEN Immersion Depth ---
h2 = px.histogram(df_23_6, x='SENImmersionDepth')#,nbins=165)
fig1.add_trace(h2.data[0], row=1, col=2)
fig1.update_xaxes(title_text="SENImmersionDepth [m]", row=1, col=2)
fig1.update_yaxes(title_text="Count", row=1, col=2)

# --- Subplot 3: Mold Width ---
h3 = px.histogram(df_23_6, x='moldWidth')
fig1.add_trace(h3.data[0], row=2, col=1)
fig1.update_xaxes(title_text="Mold Width [m]", row=2, col=1)
fig1.update_yaxes(title_text="Count", row=2, col=1)

# --- Subplot 4: Argon Flow (SEN + Stopper) ---
argon_total = df_23_6[['Argon Flow SEN', 'Argon Flow Stopper']].sum(axis=1)
h4 = px.histogram(x=argon_total)
fig1.add_trace(h4.data[0], row=2, col=2)
fig1.update_xaxes(title_text="Argon Flow SEN+STOPPER [l/min]", row=2, col=2)
fig1.update_yaxes(title_text="Count", row=2, col=2)

# --- Final Layout ---
fig1.update_layout(
    height=900,
    width=1100,
    title_text="Strand 23-6 — Parameter Distributions",
    showlegend=False
)

fig1.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/cc_parameters_histograms.html")
fig1.show()




In [0]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplot grid (2 rows × 2 columns)
fig2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Casting Speed Over Time",
        "SEN Immersion Depth Over Time",
        "Mold Width Over Time",
        "Argon Flow (SEN + Stopper) Over Time"
    ]
)

# 1️⃣ Casting Speed
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['castingSpeed'],
        mode='lines'
    ),
    row=1, col=1
)

# 2️⃣ SEN Immersion Depth
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['SENImmersionDepth'],
        mode='lines'
    ),
    row=1, col=2
)

# 3️⃣ Mold Width
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['moldWidth'],
        mode='lines'
    ),
    row=2, col=1
)

# 4️⃣ Argon Flow = SEN + Stopper
argon_total = df_23_6[['Argon Flow SEN', 'Argon Flow Stopper']].sum(axis=1)

fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=argon_total,
        mode='lines'
    ),
    row=2, col=2
)

# Axis labels
fig2.update_xaxes(title_text="Timestamp", row=1, col=1)
fig2.update_xaxes(title_text="Timestamp", row=1, col=2)
fig2.update_xaxes(title_text="Timestamp", row=2, col=1)
fig2.update_xaxes(title_text="Timestamp", row=2, col=2)

fig2.update_yaxes(title_text="Casting Speed [m/min]", row=1, col=1)
fig2.update_yaxes(title_text="SEN Immersion Depth [m]", row=1, col=2)
fig2.update_yaxes(title_text="Mold Width [m]", row=2, col=1)
fig2.update_yaxes(title_text="Argon Flow [l/min]", row=2, col=2)

# Figure layout
fig2.update_layout(
    height=1000,
    width=1200,
    title_text="Strand 23-6 — Time Series of Key Parameters",
    showlegend=False
)
fig2.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/cc_parameters_timeseriesPlots.html")
fig2.show()


In [0]:
df_23_6.shape

In [0]:
from matplotlib.colors import LogNorm# "Viridis-like" colormap with white background

white_viridis = LinearSegmentedColormap.from_list('white_viridis', [
    (0, '#ffffff'),
    (1e-20, '#440053'),
    (0.2, '#404388'),
    (0.4, '#2a788e'),
    (0.6, '#21a784'),
    (0.8, '#78d151'),
    (1, '#fde624'),
], N=256)

def using_mpl_scatter_density(fig, x, y, labels):
    ax = fig.add_subplot(1, 1, 1, projection='scatter_density')
    density = ax.scatter_density(x, y, cmap=white_viridis, norm=LogNorm(vmin=1,vmax=5000), dpi=12)
    fig.colorbar(density, label=labels)

fig3 = plt.figure(tight_layout = True)
using_mpl_scatter_density(fig3, df_23_6.moldWidth, df_23_6.castingSpeed, labels='CC snapshot counts')
plt.xlabel('mold Width, [m]')
plt.ylabel('casting Speed, [m/min]')
fig3.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/width_vs_speed_density_strd_23_6.png")
plt.show()


In [0]:
# Plotly line plot for Mold Level
fig4 = px.line(
    df_23_6,
    y=['Mold Level', 'Mold Level Sensor Left', 'Mold Level Sensor Right'],
    x=  'plainTimeStamp',
    title='Mold Level Over Time'
    )
fig4.update_layout(xaxis_title='plainTimeStamp', yaxis_title='Mold Level [m]')
fig4.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/mold_level_plot.html")
fig4.show()

In [0]:
%fs ls dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/

In [0]:
# after any sampling or filtering
df_23_6_sampled = (
    df_23_6
      .sample(frac=0.25, random_state=42)   # or whatever sampling you used
      .sort_values("plainTimeStamp")        # VERY important
      .reset_index(drop=True)
)
df_23_6_sampled.shape

In [0]:
df_23_6_sampled.head()

In [0]:
df_23_6_sampled.columns

In [0]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.dropna(subset=["plainTimeStamp"]).sort_values("plainTimeStamp").reset_index(drop=True)

# gap-compressed x-axis
df["t_idx"] = np.arange(len(df))

# Argon average (use .sum(axis=1) if you want total instead)
df["Argon Flow AVG"] = df[["Argon Flow SEN", "Argon Flow Stopper"]].mean(axis=1)

# Variables to plot (one subplot each)
series_specs = [
    ("castingSpeed",        "Casting speed"),
    ("moldWidth",           "Mold width"),
    ("SENImmersionDepth",   "SEN immersion depth"),
    ("Mold Level",          "Mold level"),
    ("Argon Flow AVG",      "Argon flow (avg of SEN+Stopper)"),
]

nrows = len(series_specs)

fig = make_subplots(
    rows=nrows, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[title for _, title in series_specs],
)

for r, (col, _) in enumerate(series_specs, start=1):
    fig.add_trace(
        go.Scatter(
            x=df["t_idx"],
            y=df[col],
            mode="lines",
            customdata=df[["plainTimeStamp"]].to_numpy(),
            hovertemplate=(
                "idx=%{x}<br>"
                "time=%{customdata[0]}<br>"
                f"{col}=%{{y}}<extra></extra>"
            ),
            name=col,
            showlegend=False,
        ),
        row=r, col=1
    )
    fig.update_yaxes(title_text=col, row=r, col=1)

# Sparse readable timestamp ticks
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig.update_xaxes(
    title_text="Time (gaps removed; hover shows real timestamp)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
    row=nrows, col=1
)

fig.update_layout(
    title="Strand#23_6 – Casting parameters (sampled and gap-compressed)",
    height=260 * nrows,
    width=1200,
)

# Save & show
fig.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_casting_parameters_subplots.html")
fig.show()


In [0]:
import pandas as pd
import plotly.express as px

df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig = px.line(
    df,
    x="t_idx",
    y=[
        "castingSpeed",
        "moldWidth",
       # "SENImmersionDepth",
       # 'Mold Level'
    ],
    title="Strand#23_6: Casting speed and Mold width (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)
# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_Vc_and_width_parameters.html"
)

fig.show()


In [0]:
df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig7 = px.line(
    df,
    x="t_idx",
    y=[
        "Mold Level", 
        "Mold Level Sensor Left", 
        "Mold Level Sensor Right"
    ],
    title="Strand#23_6: Mold Level (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig7.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig7.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)
fig7.update_layout(yaxis_title="Mold Level [mm]")
fig7.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/Argon_plot.html")
fig7.show()

In [0]:


df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig8 = px.line(
    df,
    x="t_idx",
    y=[
        'Argon Flow SEN', 
        'Argon Flow Stopper'
    ],
    title="Strand#23_6: Argon flow (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig8.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig8.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)

fig8.show()

In [0]:


df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig8 = px.line(
    df,
    x="t_idx",
    y=[
        "EMBR Current DC Left Bottom",
        "EMBR Current AC Left Master",
        "EMBR Current DC Left Master",
    ],
    title="Strand#23_6: FC Mold currents (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig8.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig8.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)

fig8.show()


In [0]:
%python
import plotly.express as px

columns_to_plot = [
    "castingSpeed",
    "moldWidth",
    "SENImmersionDepth",
    "Mold Level",
    "Argon Flow SEN",
    "Argon Flow Stopper",
    "EMBR Current DC Left Bottom",
    "EMBR Current AC Left Master",
    "EMBR Current DC Left Master"
]

# Reshape to long format
df_long = df_23_6.melt(
    id_vars=["plainTimeStamp"],
    value_vars=columns_to_plot,
    var_name="Parameter",
    value_name="Value"
)

fig = px.line(
    df_long,
    x="plainTimeStamp",
    y="Value",
    color="Parameter",
    facet_col="Parameter",
    facet_col_wrap=4,
    labels={"plainTimeStamp": "Time", "Value": "Value"},
    color_discrete_sequence=["blue"],
    facet_col_spacing=0.05,
    facet_row_spacing=0.1,
    category_orders={"plainTimeStamp": df_long["plainTimeStamp"].astype(str).unique().tolist()}
)


fig.update_layout(showlegend=False)

fig.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand1_ALL_signals_SEN.html")
fig.show()

## Data grouping

In [0]:
# Function to round values with custom logic
def custom_rounding(series, round_up=True):
    rounded_values = []
    for i in range(len(series)):
        if i > 0 and abs(series[i] - series[i-1]) <= 0.01: # CASTING_SPEED rounding difference
            rounded_value = rounded_values[-1]
        else:
            if round_up:
                rounded_value = round(series[i] * 100 + 0.5) / 100
            else:
                rounded_value = round(series[i] * 100 - 0.5) / 100
        rounded_values.append(rounded_value)
    return rounded_values
  

def filter_and_group_sequences(df, status_column, speed_column, status_value=1, diff_threshold=0.1, min_length=300):
    """
    Filters the DataFrame based on a status column and groups sequences of rows
    where the difference between consecutive values in a specified column is
    less than or equal to a threshold, retaining only groups with a minimum length.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        status_column (str): The column to filter by status.
        speed_column (str): The column to group by consecutive differences.
        status_value (int): The value to filter the status column. Default is 1.
        diff_threshold (float): The maximum allowed difference between consecutive values. Default is 0.1.
        min_length (int): The minimum length of a valid group. Default is 300.

    Returns:
        pd.DataFrame: A DataFrame containing valid groups.
    """
    # Step 1: Filter the DataFrame where the status column matches the specified value
    filtered_df = df[df[status_column] == status_value].reset_index(drop=True)

    # Step 2: Identify groups based on the difference in the speed column
    # Create a group identifier where the difference between consecutive rows is > diff_threshold
    filtered_df['group'] = (filtered_df[speed_column].diff().abs() > diff_threshold).cumsum()

    # Step 3: Group by the 'group' column and filter groups with at least the minimum length
    valid_groups = (
        filtered_df.groupby('group')
        .filter(lambda x: len(x) >= min_length)
        .reset_index(drop=True)
    )

    # Drop the temporary 'group' column if not needed
    valid_groups = valid_groups.drop(columns=['group'])

    return valid_groups

# Segment the time series into continuous chunks based on gaps in plainTimeStamp

def segment_by_time_gaps(df, tcol="plainTimeStamp", max_gap_seconds=5):
    """
    Split a time-series DataFrame into continuous segments based on time gaps.

    A new segment starts whenever the time difference between consecutive rows
    exceeds `max_gap_seconds`.

    Returns:
        df_seg: df with tcol as datetime and a new 'segment_id' column.
    """
    df_seg = df.copy()

    # ensure datetime and sort by time
    df_seg[tcol] = pd.to_datetime(df_seg[tcol])
    df_seg = df_seg.sort_values(tcol)

    # time difference in seconds
    dt = df_seg[tcol].diff().dt.total_seconds().fillna(0)

    # cumsum over "gap" indicators → segment IDs: 0,1,2,...
    df_seg["segment_id"] = (dt > max_gap_seconds).cumsum()

    return df_seg



# Function to identify groups of consecutive values within the difference threshold using a sliding window of step 1
def identify_sequences(df,
                       Vc_column,
                       window_size,
                       Vc_threshold,
                       Curr_columns=None,
                       Curr_threshold=None):
    """
    Sliding window with step 1.

    A window is classified as NORMAL if:
      - all values of Vc_column in the window are within Vc_threshold
        of the first value; AND
      - for every column in Curr_columns (if provided), all consecutive
        differences are < Curr_threshold.

    Otherwise the window is ABNORMAL.

    Returns:
        normal_groups: list of lists of indices (NORMAL windows)
        abnormal_groups: list of lists of indices (ABNORMAL windows)
    """
    normal_groups = []
    abnormal_groups = []

    i = 0
    n = len(df)

    while i <= n - window_size:
        window_df = df.iloc[i:i + window_size]
        window_idx = window_df.index

        # 1) main condition on casting speed
        Vc_window = window_df[Vc_column]
        cond_vc = np.all(np.abs(Vc_window - Vc_window.iloc[0]) <= Vc_threshold)

        # 2) additional conditions on current columns (if provided)
        cond_curr = True
        if cond_vc and Curr_columns is not None and Curr_threshold is not None:
            for col in Curr_columns:
                if col not in window_df.columns:
                    cond_curr = False
                    break
                series = window_df[col]
                # check consecutive differences
                if np.any(np.abs(series.diff().dropna()) >= Curr_threshold):
                    cond_curr = False
                    break

        is_normal = cond_vc and cond_curr

        if is_normal:
            normal_groups.append(window_idx.tolist())
            # skip past this whole stable window
            i += window_size
        else:
            abnormal_groups.append(window_idx.tolist())
            # move start forward by ONE sample and try again
            i += 1

    return normal_groups, abnormal_groups

def identify_sequences_segmented(df,
                                 Vc_column,
                                 window_size,
                                 Vc_threshold,
                                 Curr_columns=None,
                                 Curr_threshold=None,
                                 tcol="plainTimeStamp",
                                 max_gap_seconds=5,
                                 min_segment_len=None):
    """
    Apply identify_sequences within continuous time segments so that
    sliding windows never cross large time gaps.

    Args:
        df: original DataFrame
        Vc_column: name of casting speed column
        window_size: window length in samples
        Vc_threshold: threshold for "stable" speed
        Curr_columns: list of additional columns to check
        Curr_threshold: threshold for consecutive differences in those columns
        tcol: timestamp column
        max_gap_seconds: max allowed gap inside a segment
        min_segment_len: if set, skip segments shorter than this length
                         (often set to window_size)

    Returns:
        normal_groups_all: list of index lists (global indices)
        abnormal_groups_all: list of index lists (global indices)
    """
    df_seg = segment_by_time_gaps(df, tcol=tcol, max_gap_seconds=max_gap_seconds)

    normal_groups_all = []
    abnormal_groups_all = []

    for seg_id, seg_df in df_seg.groupby("segment_id"):
        if min_segment_len is not None and len(seg_df) < min_segment_len:
            continue

        n_g, a_g = identify_sequences(
            seg_df,
            Vc_column=Vc_column,
            window_size=window_size,
            Vc_threshold=Vc_threshold,
            Curr_columns=Curr_columns,
            Curr_threshold=Curr_threshold,
        )

        normal_groups_all.extend(n_g)
        abnormal_groups_all.extend(a_g)

    return normal_groups_all, abnormal_groups_all


In [0]:
df_23_6.head(5)

In [0]:
df_FCMold_ON_str23_6 = df_23_6[(df_23_6['EMBR Current AC Left Master'] != 0) & (df_23_6['EMBR Current DC Left Master'] != 0) & (df_23_6['EMBR Current DC Left Bottom'] != 0)].reset_index(drop=True)

In [0]:
(df_23_6.shape, df_FCMold_ON_str23_6.shape)

In [0]:
df_FCMold_ON_str23_6['castingSpeed'] =  custom_rounding(df_FCMold_ON_str23_6.castingSpeed, 2)

In [0]:
df_FCMold_ON_str23_6.head(10)

In [0]:
# Example usage
window_size = 300  # 6 min window size
Vc_column_str5 = 'castingSpeed'
Vc_column_str6 = 'castingSpeed'
Vc_threshold = 0.1
Curr_columns_str5 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_columns_str6 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_threshold = 50

In [0]:

Vc_column_str6 = "castingSpeed"

normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = identify_sequences_segmented(
    df_FCMold_ON_str23_6,
    Vc_column=Vc_column_str6,
    window_size=window_size,
    Vc_threshold=Vc_threshold,
    Curr_columns=Curr_columns_str6,
    Curr_threshold=Curr_threshold,
    tcol="plainTimeStamp",
    max_gap_seconds=5,          # treat gaps >5s as new segments
    min_segment_len=window_size # ignore segments shorter than one window
)

len(normal_groups_ON_str23_6), len(abnormal_groups_ON_str23_6)

In [0]:
import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
import pandas as pd
import math

df = df_FCMold_ON_str23_6
Vc_col = Vc_column_str6
window_size_vis = 300
max_windows = 12
Vc_threshold = Vc_threshold

# ---------- 1) Restrict DF to FIRST 30 MIN ONLY & SORT ----------
if "plainTimeStamp" in df.columns:
    tcol = "plainTimeStamp"
elif "plainTimestamp" in df.columns:
    tcol = "plainTimestamp"
else:
    raise ValueError("Need timestamp column.")

ts_full = df[tcol]
start_ts = ts_full.iloc[0]
end_ts_2h = start_ts + pd.Timedelta(hours=0.5)

mask_2h = (ts_full >= start_ts) & (ts_full <= end_ts_2h)
df_chunk = (
    df.loc[mask_2h]
      .copy()
      .sort_values(tcol)
      .reset_index(drop=True)
)

ts = df_chunk[tcol]
Vc = df_chunk[Vc_col]
y_all = df_chunk["castingSpeed"]

n = len(df_chunk)
idx_all = np.arange(n)

# ---------- 2) Sliding-window search on chunk ----------
windows = []

i = 0
while i <= n - window_size_vis and len(windows) < max_windows:
    window = Vc.iloc[i:i + window_size_vis]
    diffs = np.abs(window - window.iloc[0])

    bad_offsets = np.where(diffs > Vc_threshold)[0]
    if len(bad_offsets) > 0:
        first_bad = bad_offsets[0]
        break_idx = i + first_bad
        cond = False
    else:
        break_idx = None
        cond = True

    start = i
    end = i + window_size_vis
    windows.append((start, end, break_idx, cond))

    if cond:
        i += window_size_vis
    else:
        i = break_idx + 1

n_windows = len(windows)
n_cols = 2
n_rows = max(1, math.ceil(n_windows / n_cols))

# ---------- 3) Plotly figure with 2 columns ----------
titles = [
    f"Window {k} (start={start}, end={end-1}) — {'NORMAL' if cond else 'ABNORMAL'}"
    for k, (start, end, break_idx, cond) in enumerate(windows)
]
# pad titles if last row not full
if len(titles) < n_rows * n_cols:
    titles += [""] * (n_rows * n_cols - len(titles))

fig = sp.make_subplots(
    rows=n_rows,
    cols=n_cols,
    shared_xaxes=True,
    vertical_spacing=0.06,
    horizontal_spacing=0.08,
    subplot_titles=titles,
)

for idx, (start, end, break_idx, cond) in enumerate(windows):
    row = idx // n_cols + 1
    col = idx % n_cols + 1

    first_subplot = (idx == 0)
    normal_subplot = (idx == n_windows - 1)

    gray_y = y_all.copy()

    in_window = (idx_all >= start) & (idx_all < end)
    blue_mask = np.zeros(n, dtype=bool)
    black_mask = np.zeros(n, dtype=bool)
    red_mask = np.zeros(n, dtype=bool)

    if cond:
        blue_mask[in_window] = True
        gray_y[blue_mask] = np.nan
    else:
        if break_idx is not None:
            black_mask[(idx_all >= start) & (idx_all < break_idx)] = True
            red_mask[(idx_all >= break_idx) & (idx_all < end)] = True

        gray_y[in_window] = np.nan

    # gray trace
    fig.add_trace(
        go.Scatter(
            x=ts,
            y=gray_y,
            mode="lines",
            line=dict(color="gray", width=1),
            name="castingSpeed, m/min" if first_subplot else None,
            showlegend=first_subplot,
        ),
        row=row,
        col=col,
    )

    # blue: full normal windows
    if blue_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(blue_mask, np.nan),
                mode="lines",
                line=dict(color="blue", width=3),
                name="Normal window", # if first_subplot else None,
                showlegend=normal_subplot,
            ),
            row=row,
            col=col,
        )

    # black: pre-break stable region in abnormal windows
    if black_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(black_mask, np.nan),
                mode="lines",
                line=dict(color="black", width=2),
                name="normal region" if first_subplot else None,
                showlegend=first_subplot,
            ),
            row=row,
            col=col,
        )

    # red: abnormal region
    if red_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(red_mask, np.nan),
                mode="lines",
                line=dict(color="red", width=3),
                name="abnormal region" if first_subplot else None,
                showlegend=first_subplot,
            ),
            row=row,
            col=col,
        )

# layout
fig.update_layout(
    height=260 * n_rows,
    width=1300,
    title="Data grouping - Sliding Window Visualization",
)

# x-axis label only on bottom row
for c in range(1, n_cols + 1):
    fig.update_xaxes(title_text="Time", row=n_rows, col=c)

fig.show()


In [0]:
# Populate df_result with sequence's statistics
def generate_seq_statistics(df, sequences, seq_condition):
    df_result = pd.DataFrame()
    cnt = 0

    for group in sequences:
        if max(group) >= len(df):
            raise IndexError("positional indexers are out-of-bounds")
        
        temp_df = df.iloc[group]

        temp_df = temp_df.reset_index(drop=True)

        # Populate df_results with Statistics
        df_result.loc[cnt, 'Seq_Name'] = ','.join(['Seq#'+'%d' %(cnt)]) #', '.join(temp_df['filename'].apply(lambda x: os.path.basename(x)).unique())[25:29]+'_%d' %(cnt)]

        timestamps = temp_df['plainTimeStamp'].to_list()

        if timestamps:
            df_result.loc[cnt, 'Seq_time_Start'] = timestamps[0]
            #if len(timestamps) > (result_reduced[0][1] - 1):
            #    df_result.loc[cnt, 'Seq_time_End'] = timestamps[result_reduced[0][1] - 1]
            #else:
            df_result.loc[cnt, 'Seq_time_End'] = timestamps[-1]  # Use the last timestamp if the index is out of range
        else:
            df_result.loc[cnt, 'Seq_time_Start'] = None
            df_result.loc[cnt, 'Seq_time_End'] = None

        df_result.loc[cnt, 'SEN_avg [mm]'] = temp_df['SENImmersionDepth'].mean()
        df_result.loc[cnt, 'SEN_std [mm]'] = temp_df['SENImmersionDepth'].std()

       
        # Compute statistics
        df_result.loc[cnt,'CASTING_SPEED_avg [m/min]'] = temp_df['castingSpeed'].mean()
        df_result.loc[cnt,'CASTING_SPEED_std [m/min]'] = temp_df['castingSpeed'].std()
        df_result.loc[cnt,'MOLD_WIDTH_avg [m]'] = temp_df['moldWidth'].mean()
        df_result.loc[cnt,'MOLD_WIDTH_std [m]'] = temp_df['moldWidth'].std()
        df_result.loc[cnt,'min DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].min()
        df_result.loc[cnt,'mean DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].mean()
        df_result.loc[cnt,'max DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].max()
        df_result.loc[cnt,'min DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].min()
        df_result.loc[cnt,'mean DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].mean()
        df_result.loc[cnt,'max DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].max()
        df_result.loc[cnt,'min AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].min()
        df_result.loc[cnt,'mean AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].mean()
        df_result.loc[cnt,'max AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].max()
        df_result.loc[cnt, 'MOLD_LEVEL_avg [mm]'] = temp_df['Mold Level'].mean()
        df_result.loc[cnt, 'MOLD_LEVEL_std [mm]'] = temp_df['Mold Level'].std()
        df_result.loc[cnt, 'ArFlow_avg [NL/min]'] = (temp_df['Argon Flow SEN'] + temp_df['Argon Flow Stopper']).mean()
        df_result.loc[cnt, 'ArFlow_std [NL/min]'] = (temp_df['Argon Flow SEN'] + temp_df['Argon Flow Stopper']).std()
        #df_result.loc[cnt, 'SEN_avg [m]'] = temp_df['SEN'].mean()
        #df_result.loc[cnt, 'SEN_std [m]'] = temp_df['SEN'].std()

        
        df_result.loc[cnt, 'Seq_Condition'] = seq_condition
        cnt +=1
        
    return df_result



In [0]:
df_seq_normal_ON_str23_6 = generate_seq_statistics(df_FCMold_ON_str23_6, normal_groups_ON_str23_6, 'normal')

In [0]:
df_seq_normal_ON_str23_6.shape

In [0]:
df_seq_normal_ON_str23_6.head()

In [0]:
def plot_MLsigma_per_sequence(df_seq, df_FCstatus, str_MLsignal, str_label):
    # Remove underscores from the 'Seq_Name' column values
    df_seq['Seq_Name'] = df_seq['Seq_Name'].str.replace('_', '', regex=False)

    for seq in df_seq['Seq_Name'].to_list():
        # Extract the sequence info - statistics - 
        seq_info = df_seq[df_seq['Seq_Name'] == seq]
        # Remove underscores from the 'Seq_Name' column values
        #seq_info['Seq_Name'] = seq_info['Seq_Name'].str.replace('_', '', regex=False)

        # Get the start and end time as strings
        start_time = seq_info['Seq_time_Start'].iloc[0]
        end_time = seq_info['Seq_time_End'].iloc[0]

        # Slice the DataFrame based on the 'time' column
        df_temp = df_FCstatus[(df_FCstatus['plainTimeStamp'] >= start_time) & (df_FCstatus['plainTimeStamp'] <= end_time)]

        #df_ON_seq6 = df_FCMold_ON.iloc[normal_groups_ON[21]]
        # Calculate mean, min, and max values
        mean_value = df_temp[str_MLsignal].mean()
        min_value = df_temp[str_MLsignal].min()
        max_value = df_temp[str_MLsignal].max()

        # Create the line plot
        fig11 = px.line(
            df_temp,
            x="plainTimeStamp",
            y=[str_MLsignal],
            title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                str_label, seq, start_time, end_time,
                seq_info['CASTING_SPEED_avg [m/min]'].values[0],
                seq_info['min AC Current, Up [A]'].values[0],
                seq_info['mean DC Current, Up [A]'].values[0],
                seq_info['mean DC Current, Low [A]'].values[0]
            ),
)
        fig11.update_layout(title_font_size=14,  # Set the title font size
            yaxis_title="Mold Level [mm]"  # Add y-axis label
        )
        # Add horizontal lines for mean, min, and max
        fig11.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
        fig11.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
        fig11.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

        # Save the plot as a PNG file
        # Sanitize time strings for filename
        safe_start_time = start_time.strftime('%Y-%m-%d_%H-%M-%S')
        safe_end_time = end_time.strftime('%Y-%m-%d_%H-%M-%S')
       
        #fig2.write_html("../reports/figures/data_grouping/window_size_5min/%s_%s_%s_%s_Vc=%.2f_AC_up=%d_DC_up=%d_DC_low=%d.html" % (
        #        str_label, seq, safe_start_time, safe_end_time,
        #        seq_info['CASTING_SPEED_avg [m/min]'].values[0],
        #        seq_info['min AC Current, Up [A]'].values[0],
        #        seq_info['mean DC Current, Up [A]'].values[0],
        #        seq_info['mean DC Current, Low [A]'].values[0]))#, width=1200, height=600, scale=2)
        # Show the plot
        #fig11.show()

In [0]:
#plot_MLsigma_per_sequence(df_seq_normal_ON_str23_6, 
#                            df_FCMold_ON_str23_6, 
#                            str_MLsignal = "Mold Level", 
#                            str_label = 'Strd23_6')

In [0]:
df_seq_normal_ON_str23_6.head()

In [0]:
df_seq_normal_ON_str23_6.shape[0]

In [0]:
for i in range(5): #df_seq_normal_ON_str23_6.shape[0]):
    print(df_seq_normal_ON_str23_6.iloc[i]['Seq_Name'])
    # Select the 
    sequence = i

    df_ON_seq23_6 = df_FCMold_ON_str23_6.iloc[normal_groups_ON_str23_6[sequence]]
    # Calculate mean, min, and max values
    mean_value = df_ON_seq23_6["Mold Level"].mean()
    min_value = df_ON_seq23_6["Mold Level"].min()
    max_value = df_ON_seq23_6["Mold Level"].max()

    #metadata
    str_label = 'strd23_6'
    seq = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_Name']
    start_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_Start']
    end_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_End']
    Vc_seq = df_seq_normal_ON_str23_6.iloc[sequence]['CASTING_SPEED_avg [m/min]']
    ACup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['min AC Current, Up [A]']
    DCup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Up [A]']
    DClow_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Low [A]']

    # Crete the line plot
    fig10 = px.line(
        df_ON_seq23_6,
        x="plainTimeStamp",
        y=["Mold Level"],
        title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                    str_label, seq, start_time, end_time,
                    Vc_seq,
                    ACup_seq,
                    DCup_seq,
                    DClow_seq
                ),
            )

    # Add horizontal lines for mean, min, and max
    fig10.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
    fig10.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
    fig10.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

    # Save the plot as a PNG file
    #fig2.write_html("../reports/figures/data_grouping/window_size_5min/strand1_seq21_mold_level.html")#, width=1200, height=600, scale=2)
    # Show the plot
    fig10.show()

In [0]:
df_seq_normal_ON_str23_6.columns

In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_ON_str23_6

# --- Helper to build each scatter with Seq_Name in hover only ---
def make_scatter(df, x_col, x_label):
    fig_px = px.scatter(
        df,
        x=x_col,
        y="MOLD_LEVEL_std [mm]",
        hover_data=["Seq_Name"],     # show Seq_Name in hover
        labels={
            x_col: x_label,
            "MOLD_LEVEL_std [mm]": "Mold Level, [mm]"
        }
    )

    trace = fig_px.data[0]
    trace.text = None  # <-- remove text labels on points
    trace.hovertemplate = (
        "Seq_Name: %{customdata[0]}<br>"
        f"{x_label}: %{{x}}<br>"
        "Mold Level, [mm]: %{y}<extra></extra>"
    )

    return trace

# --- Create 2x2 subplot layout ---
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Vc vs Mold Level",
        "Mold Width vs Mold Level",
        "SEN vs Mold Level",
        "Argon Flow vs Mold Level"
    ]
)

# 1) Casting speed vs Mold Level
fig.add_trace(
    make_scatter(df, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]"),
    row=1, col=1
)

# 2) Mold width vs Mold Level
fig.add_trace(
    make_scatter(df, "MOLD_WIDTH_avg [m]", "Mold Width Avg [m]"),
    row=1, col=2
)

# 3) SEN vs Mold Level
fig.add_trace(
    make_scatter(df, "SEN_avg [mm]", "SEN Avg [mm]"),
    row=2, col=1
)

# 4) Argon Flow vs Mold Level
fig.add_trace(
    make_scatter(df, "ArFlow_avg [NL/min]", "Ar Flow Avg [NL/min]"),
    row=2, col=2
)

# --- Add horizontal threshold lines ---
for r in [1, 2]:
    for c in [1, 2]:
        fig.add_hline(
            y=2,
            line_dash="dash",
            line_color="green",
            annotation_text="ML fluctuation = 2 mm",
            annotation_position="top left",
            row=r, col=c
        )

# --- Axis titles ---
fig.update_xaxes(title_text="Casting Speed Avg [m/min]", row=1, col=1)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=1, col=1)

fig.update_xaxes(title_text="Mold Width Avg [m]",        row=1, col=2)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=1, col=2)

fig.update_xaxes(title_text="SEN Avg [mm]",              row=2, col=1)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=2, col=1)

fig.update_xaxes(title_text="Ar Flow Avg [NL/min]",      row=2, col=2)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=2, col=2)

# --- Layout ---
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Correlations vs Mold Level",
    showlegend=False
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_cc_parameters_and_ML_correlations.html"
)
fig.show()


In [0]:
sequence = 355

df_ON_seq23_6 = df_FCMold_ON_str23_6.iloc[normal_groups_ON_str23_6[sequence]]
    # Calculate mean, min, and max values
mean_value = df_ON_seq23_6["Mold Level"].mean()
min_value = df_ON_seq23_6["Mold Level"].min()
max_value = df_ON_seq23_6["Mold Level"].max()

    #metadata
str_label = 'strd23_6'
seq = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_Name']
start_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_Start']
end_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_End']
Vc_seq = df_seq_normal_ON_str23_6.iloc[sequence]['CASTING_SPEED_avg [m/min]']
ACup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['min AC Current, Up [A]']
DCup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Up [A]']
DClow_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Low [A]']

    # Crete the line plot
fig11 = px.line(
    df_ON_seq23_6,
    x="plainTimeStamp",
    y=["Mold Level"],
    title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                 str_label, seq, start_time, end_time,
                Vc_seq,
                ACup_seq,
                DCup_seq,
                DClow_seq
            ),
        )

    # Add horizontal lines for mean, min, and max
fig11.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
fig11.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
fig11.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

    # Save the plot as a PNG file
    #fig2.write_html("../reports/figures/data_grouping/window_size_5min/strand1_seq21_mold_level.html")#, width=1200, height=600, scale=2)
    # Show the plot
fig11.show()

In [0]:
df_seq_normal_ON_str23_6.columns

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Colormap where low values are visible (no white-on-white)
ml_cmap = LinearSegmentedColormap.from_list(
    'ml_sigma_cmap',
    [
        (0.0, '#440053'),   # dark purple  (very low σ)
        (0.3, '#404388'),
        (0.6, '#2a788e'),
        (0.8, '#21a784'),
        (0.95, '#78d151'),
        (1.0, '#fde624'),   # yellow       (very high σ)
    ],
    N=256
)

ml_std = df_seq_normal_ON_str23_6['MOLD_LEVEL_std [mm]'].to_numpy()
ml_std_max = np.nanmax(ml_std)

norm = TwoSlopeNorm(vmin=0, vcenter=2, vmax=ml_std_max)

x = df_seq_normal_ON_str23_6['MOLD_WIDTH_avg [m]'].to_numpy()
y = df_seq_normal_ON_str23_6['CASTING_SPEED_avg [m/min]'].to_numpy()

mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x = x[mask]
y = y[mask]
ml_std = ml_std[mask]

above_thr = ml_std > 2.0  # bad region

fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter: all points colored by σ, threshold-aware colormap
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge so they stand out
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=60,
    edgecolor="black",
    linewidth=0.7,
    alpha=0.95,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('MOLD_LEVEL_std [mm]')

ax.set_xlabel('Mold Width avg [m]')
ax.set_ylabel('Casting Speed avg [m/min]')
ax.set_title('Mold Width vs Casting Speed colored by Mold Level σ')

# Optional: visual threshold line in colorbar label
# (the TwoSlopeNorm already makes 2 mm visually special)
plt.show()



In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Colormap where low values are visible (no white-on-white)
ml_cmap = LinearSegmentedColormap.from_list(
    'ml_sigma_cmap',
    [
        (0.0, '#440053'),   # dark purple  (very low σ)
        (0.3, '#404388'),
        (0.6, '#2a788e'),
        (0.8, '#21a784'),
        (0.95, '#78d151'),
        (1.0, '#fde624'),   # yellow       (very high σ)
    ],
    N=256
)

ml_std = df_seq_normal_ON_str23_6['MOLD_LEVEL_std [mm]'].to_numpy()
ml_std_max = np.nanmax(ml_std)

norm = TwoSlopeNorm(vmin=0, vcenter=2, vmax=ml_std_max)

x = df_seq_normal_ON_str23_6['SEN_avg [mm]'].to_numpy()
y = df_seq_normal_ON_str23_6['ArFlow_avg [NL/min]'].to_numpy()

mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x = x[mask]
y = y[mask]
ml_std = ml_std[mask]

above_thr = ml_std > 2.0  # bad region

fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter: all points colored by σ, threshold-aware colormap
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge so they stand out
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=60,
    edgecolor="black",
    linewidth=0.7,
    alpha=0.95,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('MOLD_LEVEL_std [mm]')

ax.set_xlabel('SEN_avg [m]')
ax.set_ylabel('ArFlow_avg [NL/min]')
#ax.set_title('Mold Width vs Casting Speed colored by Mold Level σ')

# Optional: visual threshold line in colorbar label
# (the TwoSlopeNorm already makes 2 mm visually special)
plt.show()



In [0]:
ml_colorscale = [
    [0.0,  "#440053"],  # dark purple – very low σ
    [0.3,  "#404388"],
    [0.6,  "#2a788e"],
    [0.8,  "#21a784"],
    [0.95, "#78d151"],
    [1.0,  "#fde624"],  # yellow – very high σ
]


import plotly.graph_objects as go
import numpy as np
import pandas as pd

df = df_seq_normal_ON_str23_6

ml_std = df["MOLD_LEVEL_std [mm]"]
vmin = 0.0
vmax = float(ml_std.max())
threshold = 2.0

# Custom colorscale similar to the Matplotlib one
ml_colorscale = [
    [0.0,  "#440053"],
    [0.3,  "#404388"],
    [0.6,  "#2a788e"],
    [0.8,  "#21a784"],
    [0.95, "#78d151"],
    [1.0,  "#fde624"],
]

# Split data into "normal" (σ ≤ 2 mm) and "abnormal" (σ > 2 mm)
df_normal   = df[ml_std <= threshold]
df_abnormal = df[ml_std >  threshold]

fig = go.Figure()

# ---- NORMAL trace (σ ≤ 2 mm) ----
fig.add_trace(
    go.Scatter3d(
        x=df_normal['max AC Current, Up [A]'],
        y=df_normal['max DC Current, Low [A]'],
        z=df_normal['max DC Current, Up [A]'],
        mode='markers',
        marker=dict(
            size=4,
            color=df_normal["MOLD_LEVEL_std [mm]"],
            colorscale=ml_colorscale,
            cmin=vmin,
            cmax=vmax,
            coloraxis="coloraxis",
            opacity=0.8,
        ),
        name="σ ≤ 2 mm",
        hovertemplate=(
            "AC Up: %{x:.2f} A<br>"
            "DC Low: %{y:.2f} A<br>"
            "DC Up: %{z:.2f} A<br>"
            "ML σ: %{marker.color:.2f} mm<extra>Normal</extra>"
        ),
    )
)

# ---- ABNORMAL trace (σ > 2 mm), highlighted with black edge ----
fig.add_trace(
    go.Scatter3d(
        x=df_abnormal['max AC Current, Up [A]'],
        y=df_abnormal['max DC Current, Low [A]'],
        z=df_abnormal['max DC Current, Up [A]'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_abnormal["MOLD_LEVEL_std [mm]"],
            colorscale=ml_colorscale,
            cmin=vmin,
            cmax=vmax,
            coloraxis="coloraxis",
            opacity=0.95,
            line=dict(color="black", width=1),
        ),
        name="σ > 2 mm",
        hovertemplate=(
            "AC Up: %{x:.2f} A<br>"
            "DC Low: %{y:.2f} A<br>"
            "DC Up: %{z:.2f} A<br>"
            "ML σ: %{marker.color:.2f} mm<extra>Abnormal</extra>"
        ),
    )
)

# ---- Shared coloraxis (single colorbar) ----
fig.update_layout(
    coloraxis=dict(
        colorscale=ml_colorscale,
        cmin=vmin,
        cmax=vmax,
        colorbar=dict(
            title="MOLD_LEVEL σ [mm]",
            thickness=15,
            len=0.8,
            x=1.02,
            xanchor="left",
            y=0.5,
            yanchor="middle",
        ),
    ),
    scene=dict(
        xaxis_title="AC Current, Up (A)",
        yaxis_title="DC Current, Low (A)",
        zaxis_title="DC Current, Up (A)",
        aspectmode="cube",
        xaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        yaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        zaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        camera=dict(eye=dict(x=1.2, y=1.2, z=1.2)),
    ),
    width=800,
    height=600,
    margin=dict(l=20, r=120, b=20, t=50),
    title="Strand #23_6 – FC Mold G5 Currents colored by Mold Level σ",
    title_x=0.5,
    autosize=False,
)

fig.show()


# Data grouping - Disturbance detector


In [0]:
import numpy as np
import pandas as pd

# ---------- detectors from our latest approach ----------
def has_slow_drift(df_seq, value_col="Mold Level_clean", time_col="plainTimeStamp",
                   min_run_s=60.0, min_amp_mm=10.0):
    if len(df_seq) < 3:
        return False
    t = pd.to_datetime(df_seq[time_col])
    v = pd.to_numeric(df_seq[value_col], errors="coerce").to_numpy()
    mask = np.isfinite(v)
    if mask.sum() < 3:
        return False
    t = t[mask]
    v = v[mask]

    dt = t.diff().dt.total_seconds().median()
    if not np.isfinite(dt) or dt <= 0:
        return False

    dv = np.diff(v)
    sign = np.sign(dv)
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i-1]

    runs, start = [], 0
    for i in range(1, len(sign)):
        if sign[i] != sign[start]:
            runs.append((start, i-1))
            start = i
    runs.append((start, len(sign)-1))

    for i0, i1 in runs:
        if sign[i0] == 0:
            continue
        duration_s = (i1 - i0 + 1) * dt
        amp = abs(v[i1+1] - v[i0])
        if duration_s >= min_run_s and amp >= min_amp_mm:
            return True
    return False


def has_excursion_event_robust(df_seq, value_col="Mold Level", time_col="plainTimeStamp",
                               excursion_mm=8.0, min_duration_s=5.0):
    if len(df_seq) < 3:
        return False
    t = pd.to_datetime(df_seq[time_col])
    v = pd.to_numeric(df_seq[value_col], errors="coerce")

    dt = t.diff().dt.total_seconds().median()
    if not np.isfinite(dt) or dt <= 0:
        return False

    baseline = v.median()
    exc = (v - baseline).abs() > excursion_mm
    exc = exc.fillna(False).to_numpy()

    max_run, cur = 0, 0
    for b in exc:
        if b:
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 0

    return (max_run * dt) >= min_duration_s


def simple_spike_clean(series, median_win=5, jump_mm=5.0):
    s = pd.to_numeric(series, errors="coerce")
    med = s.rolling(median_win, center=True).median()
    spike = (med.diff().abs() > jump_mm)
    s2 = s.copy()
    s2[spike] = np.nan
    return s2.interpolate(limit_direction="both")

import numpy as np
import pandas as pd

def detect_transient_bump_dynamic(
    df_seq,
    value_col="Mold Level",
    time_col="plainTimeStamp",
    # dynamic threshold controls
    k_amp=8.0,            # how many "noise sigmas" defines a bump
    min_amp_mm=6.0,       # absolute minimum bump amplitude
    k_return=2.5,         # return band in sigmas
    # duration controls
    min_duration_s=5.0,
    max_duration_s=180.0, # bump should resolve within this time (tune)
    baseline_win_s=20.0   # rolling median baseline window
):
    if len(df_seq) < 10:
        return False, {}

    t = pd.to_datetime(df_seq[time_col])
    v = pd.to_numeric(df_seq[value_col], errors="coerce")

    # sampling time
    dt = t.diff().dt.total_seconds().median()
    if not np.isfinite(dt) or dt <= 0:
        return False, {}

    v = v.reset_index(drop=True)
    t = t.reset_index(drop=True)

    # rolling median baseline (slowly varying)
    win = max(5, int(round(baseline_win_s / dt)))
    baseline = v.rolling(win, center=True, min_periods=max(3, win//2)).median()

    resid = v - baseline

    # robust noise estimate using MAD of first differences (handles drifting baseline)
    dv = resid.diff().dropna()
    mad = np.median(np.abs(dv - np.median(dv)))
    sigma = 1.4826 * mad if mad > 0 else np.nan

    if not np.isfinite(sigma) or sigma == 0:
        # fallback: MAD on residual itself
        r = resid.dropna()
        mad2 = np.median(np.abs(r - np.median(r)))
        sigma = 1.4826 * mad2 if mad2 > 0 else 0.0

    amp_thresh = max(min_amp_mm, k_amp * sigma)
    return_band = max(1.0, k_return * sigma)

    # bump mask
    bump = resid.abs() >= amp_thresh
    bump = bump.fillna(False).to_numpy()

    # find contiguous bump segments
    if not bump.any():
        return False, {"sigma": sigma, "amp_thresh": amp_thresh}

    # segment runs
    idx = np.where(bump)[0]
    runs = []
    s = idx[0]
    prev = idx[0]
    for ii in idx[1:]:
        if ii == prev + 1:
            prev = ii
        else:
            runs.append((s, prev))
            s = prev = ii
    runs.append((s, prev))

    # evaluate each run: duration + peak prominence + return to baseline
    for a, b in runs:
        dur = (b - a + 1) * dt
        if dur < min_duration_s or dur > max_duration_s:
            continue

        peak = np.nanmax(np.abs(resid.iloc[a:b+1]))
        if peak < amp_thresh:
            continue

        # "return" check: after event, do we get back close to baseline?
        after = resid.iloc[b+1:min(len(resid), b+1+int(round(max_duration_s/dt)))]
        if len(after) == 0:
            continue

        returned = np.nanmin(np.abs(after)) <= return_band
        if returned:
            info = {
                "sigma": sigma,
                "amp_thresh": amp_thresh,
                "return_band": return_band,
                "run_start_idx": a,
                "run_end_idx": b,
                "duration_s": dur,
                "peak_mm": float(peak),
            }
            return True, info

    return False, {"sigma": sigma, "amp_thresh": amp_thresh, "return_band": return_band}

def detect_excursion_event_robust(
    seg,
    value_col="Mold Level",
    time_col="plainTimeStamp",
    excursion_mm=8.0,
    min_duration_s=5.0
):
    y = seg[value_col].values
    t = pd.to_datetime(seg[time_col]).values

    baseline = np.median(y)
    resid = y - baseline

    mask = np.abs(resid) >= excursion_mm

    if not mask.any():
        return False, {}

    # find contiguous runs
    idx = np.where(mask)[0]
    runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)

    for run in runs:
        duration = (t[run[-1]] - t[run[0]]) / np.timedelta64(1, "s")
        if duration >= min_duration_s:
            return True, {
                "start_idx": int(run[0]),
                "end_idx": int(run[-1]),
                "duration_s": float(duration),
                "peak_mm": float(np.max(np.abs(resid[run]))),
            }

    return False, {}

def detect_slow_drift_event(
    seg,
    value_col="Mold Level",
    time_col="plainTimeStamp",
    min_run_s=60.0,
    min_total_amp_mm=10.0
):
    """
    Detect a slow drift: a sustained monotonic trend with sufficient duration
    and total amplitude.
    """
    y = seg[value_col].values
    t = pd.to_datetime(seg[time_col]).values

    if len(y) < 3:
        return False, {}

    dy = np.diff(y)

    # Ignore near-zero steps (noise)
    eps = 0.05
    sign = np.sign(dy)
    sign[np.abs(dy) < eps] = 0

    run_start = None
    current_sign = 0

    for i in range(len(sign)):
        if sign[i] == 0:
            continue

        if current_sign == 0:
            # start a new run
            current_sign = sign[i]
            run_start = i
            continue

        if sign[i] != current_sign:
            # end current run
            run_end = i
            duration = (t[run_end] - t[run_start]) / np.timedelta64(1, "s")
            amplitude = abs(y[run_end] - y[run_start])

            if duration >= min_run_s and amplitude >= min_total_amp_mm:
                return True, {
                    "start_idx": int(run_start),
                    "end_idx": int(run_end),
                    "duration_s": float(duration),
                    "amplitude_mm": float(amplitude),
                    "direction": "up" if current_sign > 0 else "down",
                }

            # reset
            current_sign = sign[i]
            run_start = i

    return False, {}


def detect_high_variability_event(
    seg,
    value_col="Mold Level",
    time_col="plainTimeStamp",
    # thresholds
    range_mm=10.0,              # peak-to-peak range threshold
    frac_outside=0.10,          # fraction of samples outside band
    band_mm=4.0,                # band around baseline
):
    y = pd.to_numeric(seg[value_col], errors="coerce").to_numpy()
    if np.isfinite(y).sum() < 10:
        return False, {}

    baseline = np.nanmedian(y)
    resid = y - baseline

    # 1) peak-to-peak range
    ptp = np.nanmax(y) - np.nanmin(y)

    # 2) fraction outside baseline band
    frac = np.nanmean(np.abs(resid) >= band_mm)

    flag = (ptp >= range_mm) or (frac >= frac_outside)

    return bool(flag), {
        "baseline": float(baseline),
        "ptp_mm": float(ptp),
        "frac_outside": float(frac),
        "band_mm": float(band_mm),
    }


def _unique_quality(seg, col):
    if col not in seg.columns:
        return np.nan

    vals = (
        seg[col]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    if len(vals) == 0:
        return np.nan
    elif len(vals) == 1:
        return vals[0]
    else:
        return "MULTIPLE: " + " | ".join(sorted(vals))

# ---------- combined per-seq statistics + disturbance flags ----------
import numpy as np
import pandas as pd

def generate_seq_statistics_with_disturbance(
    df,
    sequences,
    seq_condition,
    # ---- column names (match your raw DF) ----
    tcol="plainTimeStamp",
    quality_col="Quality casting",
    vc_col="castingSpeed",
    mold_width_col="moldWidth",
    sen_col="SENImmersionDepth",
    mold_col="Mold Level",
    ar_sen_col="Argon Flow SEN",
    ar_stop_col="Argon Flow Stopper",
    dc_low_col="EMBR Current DC Left Bottom",
    dc_up_col="EMBR Current DC Left Master",
    ac_up_col="EMBR Current AC Left Master",
    # ---- disturbance thresholds ----
    excursion_mm=8.0,
    excursion_min_duration_s=5.0,
    slow_min_run_s=60.0,
    slow_min_amp_mm=10.0,
    # bump detector params (passed through)
    bump_kwargs=None,
    # variability detector params
    var_range_mm=10.0,
    var_band_mm=4.0,
    var_frac_outside=0.10,
    # ---- robustness ----
    enforce_required_cols=False,   # if True -> raise if any missing, else fill NaN
):
    bump_kwargs = bump_kwargs or {}

    # Work on a copy; ensure datetime once
    df2 = df.copy()
    df2[tcol] = pd.to_datetime(df2[tcol], errors="coerce")

    def _series(seg, c):
        if c in seg.columns:
            return seg[c]
        if enforce_required_cols:
            raise KeyError(f"Missing required column: {c}")
        return pd.Series([np.nan] * len(seg))

    def _mean(seg, c): return float(_series(seg, c).mean())
    def _std(seg, c):  return float(_series(seg, c).std(ddof=1))
    def _min(seg, c):  return float(_series(seg, c).min())
    def _max(seg, c):  return float(_series(seg, c).max())

    rows = []

    for cnt, group in enumerate(sequences):
        if len(group) == 0:
            continue
        if max(group) >= len(df2):
            raise IndexError("positional indexers are out-of-bounds")

        seg = df2.iloc[group].copy().reset_index(drop=True)

        # --- timestamps ---
        ts = seg[tcol].to_list()
        t_start = ts[0] if ts else None
        t_end   = ts[-1] if ts else None

        # --- disturbance detectors ---
        has_exc, exc_info = detect_excursion_event_robust(
            seg,
            value_col=mold_col,
            time_col=tcol,
            excursion_mm=excursion_mm,
            min_duration_s=excursion_min_duration_s
        )

        has_slow, slow_info = detect_slow_drift_event(
            seg,
            value_col=mold_col,
            time_col=tcol,
            min_run_s=slow_min_run_s,
            min_total_amp_mm=slow_min_amp_mm
        )

        has_bump, bump_info = detect_transient_bump_dynamic(
            seg,
            value_col=mold_col,
            time_col=tcol,
            **bump_kwargs
        )

        has_var, var_info = detect_high_variability_event(
            seg,
            value_col=mold_col,
            time_col=tcol,
            range_mm=var_range_mm,
            band_mm=var_band_mm,
            frac_outside=var_frac_outside
        )

        has_dist = bool(has_exc or has_slow or has_bump or has_var)

        # --- Argon total ---
        ar_total = _series(seg, ar_sen_col) + _series(seg, ar_stop_col)

        # --- Build row ---
        row = {
            # identifiers
            "Seq_Name": f"Seq#{cnt}",
            "Seq_time_Start": t_start,
            "Seq_time_End": t_end,
            "Seq_Condition": seq_condition,

            # SEN stats
            "SEN_avg [mm]": _mean(seg, sen_col),
            "SEN_std [mm]": _std(seg, sen_col),

            # casting speed stats
            "CASTING_SPEED_avg [m/min]": _mean(seg, vc_col),
            "CASTING_SPEED_std [m/min]": _std(seg, vc_col),

            # mold width stats
            "MOLD_WIDTH_avg [m]": _mean(seg, mold_width_col),
            "MOLD_WIDTH_std [m]": _std(seg, mold_width_col),

            # DC current low stats
            "min DC Current, Low [A]": _min(seg, dc_low_col),
            "mean DC Current, Low [A]": _mean(seg, dc_low_col),
            "max DC Current, Low [A]": _max(seg, dc_low_col),

            # DC current up stats
            "min DC Current, Up [A]": _min(seg, dc_up_col),
            "mean DC Current, Up [A]": _mean(seg, dc_up_col),
            "max DC Current, Up [A]": _max(seg, dc_up_col),

            # AC current up stats
            "min AC Current, Up [A]": _min(seg, ac_up_col),
            "mean AC Current, Up [A]": _mean(seg, ac_up_col),
            "max AC Current, Up [A]": _max(seg, ac_up_col),

            # mold level stats
            "MOLD_LEVEL_avg [mm]": _mean(seg, mold_col),
            "MOLD_LEVEL_std [mm]": _std(seg, mold_col),

            # argon flow stats
            "ArFlow_avg [NL/min]": float(ar_total.mean()) if len(ar_total) else np.nan,
            "ArFlow_std [NL/min]": float(ar_total.std(ddof=1)) if len(ar_total) else np.nan,

            # Steel Grade
            "Quality casting": _unique_quality(seg, quality_col),


            # disturbance flags
            "has_excursion_event": bool(has_exc),
            "has_slow_drift": bool(has_slow),
            "has_transient_bump": bool(has_bump),
            "has_high_variability": bool(has_var),
            "has_disturbance": bool(has_dist),

            # variability diagnostics
            "ptp_mm": var_info.get("ptp_mm", np.nan),
            "frac_outside_band": var_info.get("frac_outside", np.nan),

            # excursion diagnostics
            "excursion_peak_mm": exc_info.get("peak_mm", np.nan),
            "excursion_duration_s": exc_info.get("duration_s", np.nan),

            # slow drift diagnostics
            "slow_drift_amp_mm": slow_info.get("amplitude_mm", np.nan),
            "slow_drift_duration_s": slow_info.get("duration_s", np.nan),

            # bump diagnostics
            "bump_peak_mm": bump_info.get("peak_mm", np.nan),
            "bump_duration_s": bump_info.get("duration_s", np.nan),
            "sigma_noise": bump_info.get("sigma", np.nan),
        }

        rows.append(row)

    return pd.DataFrame(rows)


In [0]:
# 3) Usage
window_size = 300
Vc_threshold = 0.1
Vc_column_str6 = "castingSpeed"
Curr_columns_str6 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_threshold = 50

#df_seq_str23_6, normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = (
#    identify_sequences_segmented_with_disturbance_flags(
#        df_FCMold_ON_str23_6,
#        Vc_column=Vc_column_str6,
#        window_size=window_size,
##        Vc_threshold=Vc_threshold,
#        Curr_columns=Curr_columns_str6,
#        Curr_threshold=Curr_threshold,
#        tcol="plainTimeStamp",
#        max_gap_seconds=5,
#        min_segment_len=window_size,
#        mold_col="Mold Level",
#        excursion_mm=8.0,
#        excursion_min_duration_s=5.0,
#        slow_min_run_s=60.0,
#        slow_min_amp_mm=10.0,
#    )
#)

#df_seq_str23_6.head()


In [0]:
df_FCMold_ON_str23_6.head(5)

In [0]:
Vc_column_str6 = "castingSpeed"

normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = identify_sequences_segmented(
    df_FCMold_ON_str23_6,
    Vc_column=Vc_column_str6,
    window_size=window_size,
    Vc_threshold=Vc_threshold,
    Curr_columns=Curr_columns_str6,
    Curr_threshold=Curr_threshold,
    tcol="plainTimeStamp",
    max_gap_seconds=5,          # treat gaps >5s as new segments
    min_segment_len=window_size # ignore segments shorter than one window
)

len(normal_groups_ON_str23_6), len(abnormal_groups_ON_str23_6)

In [0]:
df_seq_normal_ON_str23_6 = generate_seq_statistics_with_disturbance(
    df=df_FCMold_ON_str23_6,
    sequences=normal_groups_ON_str23_6,
    seq_condition="ON",
    excursion_mm=8.0,
    excursion_min_duration_s=5.0,
    slow_min_run_s=60.0,
    slow_min_amp_mm=10.0
)


In [0]:
df_seq_normal_ON_str23_6.head(5)

In [0]:
def classify_disturbance(row):
    flags = []
    if row["has_excursion_event"]:
        flags.append("Excursion")
    if row["has_high_variability"]:
        flags.append("High variability")
    if row.get("has_transient_bump", False):
        flags.append("Transient bump")
    if row.get("has_slow_drift", False):
        flags.append("Slow drift")

    return "Normal" if not flags else " + ".join(flags)

def assign_disturbance_type(row):
    """
    Assign a human-readable disturbance type based on flags.
    Priority is intentional: excursion > slow drift > bump > variability > normal
    """

    flags = []

    if row.get("has_excursion_event", False):
        flags.append("Excursion")

    if row.get("has_slow_drift", False):
        flags.append("Slow drift")

    if row.get("has_transient_bump", False):
        flags.append("Transient bump")

    if row.get("has_high_variability", False):
        flags.append("High variability")

    if not flags:
        return "Normal"

    return " + ".join(flags)


df_seq_normal_ON_str23_6 = df_seq_normal_ON_str23_6.copy()
df_seq_normal_ON_str23_6["disturbance_type"] = (
    df_seq_normal_ON_str23_6.apply(assign_disturbance_type, axis=1)
)
df_seq_normal_ON_str23_6["disturbance_type"].value_counts()


In [0]:
df_seq_normal_ON_str23_6.head(5)

In [0]:

disturbance_order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + Transient bump + High variability",
]
disturbance_cols = [
    "has_disturbance",
    "has_excursion_event",
    "has_slow_drift",
    "has_transient_bump",
    "has_high_variability",
]


df_seq_normal_ON_str23_6["disturbance_type"] = pd.Categorical(
    df_seq_normal_ON_str23_6["disturbance_type"],
    categories=disturbance_order,
    ordered=True
)


df_seq_normal_ON_str23_6[disturbance_cols].sum().to_frame("count")


In [0]:
df_seq = df_seq_normal_ON_str23_6.copy()

# quick overlap table
pd.crosstab(df_seq["has_excursion_event"], df_seq["has_high_variability"], margins=True)


In [0]:
df_seq_normal_ON_str23_6["disturbance_type"].value_counts()


In [0]:
df_seq_normal_ON_str23_6["disturbance_type"].unique()


In [0]:
df_seq = df_seq_normal_ON_str23_6.copy()

def classify_disturbance(row):
    flags = []
    if row.get("has_excursion_event", False):   flags.append("Excursion")
    if row.get("has_slow_drift", False):        flags.append("Slow drift")
    if row.get("has_transient_bump", False):    flags.append("Transient bump")
    if row.get("has_high_variability", False):  flags.append("High variability")
    return "Normal" if not flags else " + ".join(flags)

df_seq["disturbance_type"] = df_seq.apply(classify_disturbance, axis=1)
df_seq["disturbance_type"].value_counts()


In [0]:
import plotly.express as px

fig = px.box(
    df_seq,
    x="CASTING_SPEED_avg [m/min]",
    y="MOLD_LEVEL_std [mm]",
    color="disturbance_type",
    points="all",
    hover_data=["Seq_Name", "disturbance_type"],
    title="Casting Speed vs Mold Level σ — by disturbance type",
    labels={
        "CASTING_SPEED_avg [m/min]": "Casting Speed Avg [m/min]",
        "MOLD_LEVEL_std [mm]": "Mold Level σ [mm]",
        "disturbance_type": "Disturbance type",
    },
)

# threshold
fig.add_hline(
    y=2.0,
    line_dash="dash",
    line_color="green",
    annotation_text="σ = 2 mm (stability threshold)",
    annotation_position="top right",
)
fig.update_layout(
    boxmode="overlay",
    height=700,
    width=1150,

    legend=dict(
        title="Disturbance type",
        x=0.02,                 # inside, left
        y=0.98,                 # inside, top
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="black",
        borderwidth=1,
        traceorder="reversed"   # reverse legend order
    )
)

fig.update_annotations(visible=False)
# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_casting_speed_and_MLsigma_correlations.html"
)
fig.show()


In [0]:
import pandas as pd
import plotly.express as px

#dfp = df_seq_normal_ON_str23_6.copy()
dfp = df_seq

# 1) Make sure disturbance_type exists (if already created, you can skip this block)
def assign_disturbance_type(row):
    flags = []
    if row.get("has_excursion_event", False):   flags.append("Excursion")
    if row.get("has_slow_drift", False):        flags.append("Slow drift")
    if row.get("has_transient_bump", False):    flags.append("Transient bump")
    if row.get("has_high_variability", False):  flags.append("High variability")
    return "Normal" if not flags else " + ".join(flags)

if "disturbance_type" not in dfp.columns:
    dfp["disturbance_type"] = dfp.apply(assign_disturbance_type, axis=1)

# 2) Round casting speed for grouping (same idea as your example)
dfp["CASTING_SPEED_avg_rounded"] = dfp["CASTING_SPEED_avg [m/min]"].round(1)

# Optional: enforce a clean legend order (adjust to what exists in your data)
disturbance_order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + High variability",
    "Excursion + Transient bump + High variability",
]
present = [c for c in disturbance_order if c in set(dfp["disturbance_type"])]
# add any unexpected categories at the end
present += [c for c in dfp["disturbance_type"].unique() if c not in present]

# 3) Boxplot with points overlay, colored by disturbance type
fig = px.box(
    dfp,
    x="CASTING_SPEED_avg_rounded",
    y="MOLD_LEVEL_std [mm]",
    color="disturbance_type",
    category_orders={"disturbance_type": present},
    points="all",
    hover_data=["Seq_Name", "disturbance_type"],
    labels={
        "CASTING_SPEED_avg_rounded": "Casting Speed Avg [m/min] (rounded)",
        "MOLD_LEVEL_std [mm]": "Mold Level σ [mm]",
        "disturbance_type": "Disturbance type",
    },
    title="Strand#23-6 — Casting Speed vs Mold Level σ (by disturbance type)",
)

# 4) Threshold line (σ = 2 mm)
fig.add_hline(
    y=2.0,
    line_dash="dash",
    line_color="green",
    annotation_text="Stability threshold (σ = 2 mm)",
    annotation_position="top right",
)

# 5) Layout + legend inside the plot
fig.update_layout(
    boxmode="overlay",
    height=700,
    width=1150,
    legend=dict(
        title="Disturbance type",
        x=0.02, y=0.98,
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="black",
        borderwidth=1,
        traceorder="reversed",
    ),
)

# If you previously added annotations elsewhere, hide them
fig.update_annotations(visible=False)

fig.show()


In [0]:
df_seq_normal_process = df_seq_normal_ON_str23_6[~df_seq_normal_ON_str23_6["has_disturbance"]]


In [0]:
df_seq_normal_process.head(5)

In [0]:
df_seq_normal_process.to_csv("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_seq_normal_summary.csv", index=False )


In [0]:
import numpy as np
import pandas as pd

df_seq = df_seq_normal_ON_str23_6.copy()

def classify_disturbance(row):
    flags = []
    if bool(row.get("has_excursion_event", False)):   flags.append("Excursion")
    if bool(row.get("has_high_variability", False)):  flags.append("High variability")
    if bool(row.get("has_transient_bump", False)):    flags.append("Transient bump")
    if bool(row.get("has_slow_drift", False)):        flags.append("Slow drift")
    return "Normal" if not flags else " + ".join(flags)

df_seq["disturbance_type"] = df_seq.apply(classify_disturbance, axis=1)

# pick one example index per type
# option A (simple): first occurrence
seq_idx_by_type = df_seq.groupby("disturbance_type").head(1).reset_index()["index"].to_dict()

# Better option B: pick the "strongest" example per type (more illustrative)
# Uncomment this if you want strongest by ptp_mm
# seq_idx_by_type = (
#     df_seq.sort_values(["disturbance_type", "ptp_mm"], ascending=[True, False])
#           .groupby("disturbance_type")
#           .head(1)
#           .reset_index()
#           .set_index("disturbance_type")["index"]
#           .to_dict()
# )

seq_idx_by_type


In [0]:
import plotly.graph_objects as go
import plotly.subplots as sp

# keys must be disturbance type NAMES (strings)
# e.g.:
seq_idx_by_type = {
     "Normal": 127,
     "High variability": 950,
     "Excursion": 956,
     "Excursion + Transient bump + High variability": 332
 }
disturbance_label_map = {
    0: "Normal",
    1: "High variability",
    2: "Excursion",
    3: "Excursion + Transient bump + High variability",
}

types = list(seq_idx_by_type.keys())


nrows = len(types)

fig = sp.make_subplots(
    rows=nrows,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=[
    f"Disturbance type: {disturbance_label_map.get(t, str(t))}  |  Sequence index: {seq_idx_by_type[t]}"
    for t in types
  ],

)

for r, dtype in enumerate(types, start=1):
    seq_idx = seq_idx_by_type[dtype]

    # raw sequence data
    df_seq_raw = df_FCMold_ON_str23_6.iloc[
        normal_groups_ON_str23_6[seq_idx]
    ].copy()

    mean_value = df_seq_raw["Mold Level"].mean()
    min_value  = df_seq_raw["Mold Level"].min()
    max_value  = df_seq_raw["Mold Level"].max()

    # Mold Level signal
    fig.add_trace(
        go.Scatter(
            x=df_seq_raw["plainTimeStamp"],
            y=df_seq_raw["Mold Level"],
            mode="lines",
            name=dtype,
            hovertemplate=(
                "Time=%{x}<br>"
                "Mold Level=%{y:.2f} mm<br>"
                f"Type={dtype}"
                "<extra></extra>"
            ),
        ),
        row=r,
        col=1,
    )

    # Mean / Min / Max reference lines
    fig.add_hline(
        y=mean_value,
        line_dash="dash",
        line_color="green",
        row=r,
        col=1,
    )
    fig.add_hline(
        y=min_value,
        line_dash="dot",
        line_color="blue",
        row=r,
        col=1,
    )
    fig.add_hline(
        y=max_value,
        line_dash="dot",
        line_color="red",
        row=r,
        col=1,
    )

    fig.update_yaxes(title_text="Mold Level [mm]", row=r, col=1)

fig.update_xaxes(title_text="Time", row=nrows, col=1)

fig.update_layout(
    height=320 * nrows,
    width=1200,
    title="Representative sequences per disturbance type",
    showlegend=False,
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_seq_groupings_by_disturbance_type.html")

fig.show()


In [0]:
import numpy as np
import matplotlib.pyplot as plt

dfp = df_seq_normal_ON_str23_6.copy()

# --- data vectors ---
x = dfp["MOLD_WIDTH_avg [m]"].to_numpy()
y = dfp["CASTING_SPEED_avg [m/min]"].to_numpy()
ml_std = dfp["MOLD_LEVEL_std [mm]"].to_numpy()

# categorical color = disturbance type (must exist)
dtype = dfp["disturbance_type"].astype(str).to_numpy()

# filter finite
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x, y, ml_std, dtype = x[mask], y[mask], ml_std[mask], dtype[mask]

above_thr = ml_std > 2.0

# --- choose a stable order for legend (optional) ---
order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + High variability",
    "Excursion + Transient bump + High variability",
]
present = [c for c in order if c in set(dtype)]
# add any unexpected categories at the end
present += [c for c in np.unique(dtype) if c not in present]

# --- map categories -> colors using matplotlib default cycle ---
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_map = {cat: colors[i % len(colors)] for i, cat in enumerate(present)}
point_colors = np.array([color_map[c] for c in dtype])

# --- plot ---
fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter (all points)
ax.scatter(
    x, y,
    c=point_colors,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge (same colors, just outlined)
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=point_colors[above_thr],
    s=70,
    edgecolor="black",
    linewidth=0.9,
    alpha=0.95,
)

# --- legend (disturbance types) ---
handles = [
    plt.Line2D([0], [0], marker="o", color="none",
               markerfacecolor=color_map[cat],
               markersize=8, label=cat)
    for cat in present
]
ax.legend(
    handles=handles,
    title="Disturbance type",
    loc="best",
    frameon=True,
)

ax.set_xlabel("Mold Width avg [m]")
ax.set_ylabel("Casting Speed avg [m/min]")
ax.set_title("Mold Width vs Casting Speed colored by disturbance type\n(black outline: MOLD_LEVEL_std > 2 mm)")

# Save & show
fig.savefig(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_Vc_MoldWidth_sequences_colored.png")
plt.show()


In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# --- filter to NORMAL sequences only ---
dfp = df_seq_normal_ON_str23_6[
    df_seq_normal_ON_str23_6["disturbance_type"] == "Normal"
].copy()

# --- custom colormap (same as your reference) ---
ml_cmap = LinearSegmentedColormap.from_list(
    "ml_sigma_cmap",
    [
        (0.0, "#440053"),   # dark purple (very low σ)
        (0.3, "#404388"),
        (0.6, "#2a788e"),
        (0.8, "#21a784"),
        (0.95, "#78d151"),
        (1.0, "#fde624"),   # yellow (very high σ)
    ],
    N=256,
)

# --- data vectors ---
x = dfp["MOLD_WIDTH_avg [m]"].to_numpy()
y = dfp["CASTING_SPEED_avg [m/min]"].to_numpy()
ml_std = dfp["MOLD_LEVEL_std [mm]"].to_numpy()

# keep finite only
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x, y, ml_std = x[mask], y[mask], ml_std[mask]

# threshold
above_thr = ml_std > 2.0
ml_std_max = np.nanmax(ml_std)

# threshold-aware normalization
norm = TwoSlopeNorm(vmin=0, vcenter=2.0, vmax=ml_std_max)

# --- plot ---
fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) base scatter (Normal only, colored by σ)
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.85,
)

# 2) highlight σ > 2 mm with black outline
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=70,
    edgecolor="black",
    linewidth=0.9,
    alpha=0.95,
)

# colorbar
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("MOLD_LEVEL_std [mm]")
cbar.ax.axhline(2.0, color="black", linewidth=1)
cbar.ax.text(
    1.05, 2.0, "σ = 2 mm",
    transform=cbar.ax.get_yaxis_transform(),
    va="center"
)

# labels & title
ax.set_xlabel("Mold Width avg [m]")
ax.set_ylabel("Casting Speed avg [m/min]")
ax.set_title("Normal sequences only\nColored by Mold Level σ (black outline: σ > 2 mm)")

plt.show()


In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helpers
# -------------------------
def make_box_points_and_meanline(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace, mean_line_trace)
    """
    x_round_name = f"{x_col}__rounded"
    df = df.copy()
    df[x_round_name] = df[x_col].round(round_to)

    # -------------------------
    # Box trace
    # -------------------------
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        boxmean=False,
        marker=dict(color=color),
        line=dict(color=color),
        opacity=0.45,
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        )
    )

    # -------------------------
    # Scatter trace
    # -------------------------
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.8),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False
    )

    # -------------------------
    # Mean line (per bin)
    # -------------------------
    mean_df = (
        df.groupby(x_round_name, as_index=False)["MOLD_LEVEL_std [mm]"]
        .mean()
        .sort_values(x_round_name)
    )

    mean_line = go.Scatter(
        x=mean_df[x_round_name],
        y=mean_df["MOLD_LEVEL_std [mm]"],
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=6),
        name="Mean trend",
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mean Mold Level σ: %{y:.3f} mm<extra></extra>"
        )
    )

    return box, scatter, mean_line


def make_box_and_points(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace) for a given x_col vs Mold Level std.
    Boxes are grouped by rounded x to create meaningful distributions.
    """
    x_round_name = f"{x_col}__rounded"
    df[x_round_name] = df[x_col].round(round_to)

    # Box trace (distribution per rounded bin)
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        name=None,
        boxmean="sd",
        marker=dict(color=color, opacity=0.85),
        line=dict(color=color, width=1),
        opacity=0.55,               # box opacity
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        )
    )

    # Scatter trace (points on top)
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.85),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False,
        name=None
    )

    return box, scatter

# -------------------------
# Subplot layout (2x2)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Vc vs Mold Level σ (Normal only)",
        "Mold Width vs Mold Level σ (Normal only)",
        "SEN vs Mold Level σ (Normal only)",
        "Argon Flow vs Mold Level σ (Normal only)",
    ]
)

panels = [
    (1, 1, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]"),
    (1, 2, "MOLD_WIDTH_avg [m]",        "Mold Width Avg [m]"),
    (2, 1, "SEN_avg [mm]",              "SEN Avg [mm]"),
    (2, 2, "ArFlow_avg [NL/min]",       "Ar Flow Avg [NL/min]"),
]

for r, c, xcol, xlabel in panels:
    box_tr, sc_tr, mean_tr = make_box_points_and_meanline(
    df, xcol, xlabel, round_to=1, color="#bdb76b"
    )

    fig.add_trace(box_tr, row=r, col=c)
    fig.add_trace(sc_tr, row=r, col=c)
    fig.add_trace(mean_tr, row=r, col=c)


    # axis titles
    fig.update_xaxes(title_text=f"{xlabel} (rounded)", row=r, col=c)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=r, col=c)

# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Normal process only: distributions (box + points)",
    boxmode="overlay",
    showlegend=False,
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_cc_parameters_and_ML_correlations_NORMAL_only_boxpoints.html"
)
fig.show()


In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

YCOL = "MOLD_LEVEL_std [mm]"

# -------------------------
# Helpers
# -------------------------
def add_box_points_mean(fig, df, x_col, x_label, row, col, bin_size, color="#bdb76b"):
    """
    Box + points + mean line using coarse bins.
    bin_size is in the units of x_col (e.g., 0.1 m/min, 0.05 m).
    """
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    # Bin x into coarse categories (works better than rounding for continuous-ish vars)
    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # 1) Boxplot
    fig.add_trace(
        go.Box(
            x=d[x_bin],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.45,
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                f"Mold Level σ [mm]: %{{y:.3f}}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    # 2) Points
    fig.add_trace(
        go.Scatter(
            x=d[x_bin],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label} (bin): %{{x}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # 3) Mean trend per bin
    mean_df = (
        d.groupby(x_bin, as_index=False)[YCOL]
        .mean()
        .sort_values(x_bin)
    )

    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color="white", width=2),
            marker=dict(size=6, color="white"),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label} (binned, Δ={bin_size})", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_scatter_mean(fig, df, x_col, x_label, row, col, bin_size, point_color="#bdb76b", line_color="white"):
    """
    Scatter + mean trend line only (no box). Uses bins to stabilize the mean line.
    """
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # 1) Scatter (raw x, not binned) for truthful geometry
    fig.add_trace(
        go.Scatter(
            x=d[x_col],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=point_color, size=6, opacity=0.55),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label}: %{{x:.4g}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # 2) Mean trend (computed on binned x, plotted at binned x)
    mean_df = (
        d.groupby(x_bin, as_index=False)[YCOL]
        .mean()
        .sort_values(x_bin)
    )

    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label}", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


# -------------------------
# Subplots (2×2)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Casting speed: box + points + mean (best)",
        "Mold width: box + points + mean (coarse bins)",
        "SEN: scatter + mean trend (no box)",
        "Argon flow: scatter + mean trend (no box)",
    ]
)

# 1) Casting speed: discrete setpoints → box works well (bin ~ 0.1 m/min)
add_box_points_mean(
    fig, df,
    x_col="CASTING_SPEED_avg [m/min]",
    x_label="Casting Speed Avg [m/min]",
    row=1, col=1,
    bin_size=0.1
)

# 2) Mold width: coarse bins needed (e.g. 0.05 m)
add_box_points_mean(
    fig, df,
    x_col="MOLD_WIDTH_avg [m]",
    x_label="Mold Width Avg [m]",
    row=1, col=2,
    bin_size=0.05
)

# 3) SEN: continuous → no box, mean trend stabilised with small bins (e.g. 0.001 m ~ 1 mm)
# Your SEN column label says [mm] but values look like meters (0.16–0.22).
# If it's meters, 0.001 = 1 mm. If it's truly mm, change bin_size accordingly.
add_scatter_mean(
    fig, df,
    x_col="SEN_avg [mm]",
    x_label="SEN Avg",
    row=2, col=1,
    bin_size=0.001
)

# 4) Argon flow: continuous → no box, mean trend stabilised with bins (e.g. 0.5 NL/min)
add_scatter_mean(
    fig, df,
    x_col="ArFlow_avg [NL/min]",
    x_label="Ar Flow Avg [NL/min]",
    row=2, col=2,
    bin_size=0.5
)

# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Normal process only: recommended summaries",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_6_recommended_correlations_NORMAL_only.html"
)
fig.show()


In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()   # aggregated df (per-sequence)
YCOL = "MOLD_LEVEL_std [mm]"
QCOL = "Quality casting"

# -------------------------
# Helpers
# -------------------------
def add_box_points_mean(fig, df, x_col, x_label, row, col, bin_size, color="#bdb76b"):
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # Box
    fig.add_trace(
        go.Box(
            x=d[x_bin],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.45,
            showlegend=False,
        ),
        row=row, col=col
    )

    # Points
    fig.add_trace(
        go.Scatter(
            x=d[x_bin],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label} (bin): %{{x}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # Mean line per bin
    mean_df = d.groupby(x_bin, as_index=False)[YCOL].mean().sort_values(x_bin)
    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color="black", width=2),
            marker=dict(size=6, color="black"),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label} (binned, Δ={bin_size})", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_scatter_mean(fig, df, x_col, x_label, row, col, bin_size, point_color="#bdb76b", line_color="black"):
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # Scatter (raw x)
    fig.add_trace(
        go.Scatter(
            x=d[x_col],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=point_color, size=6, opacity=0.55),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label}: %{{x:.4g}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # Mean trend (computed on binned x)
    mean_df = d.groupby(x_bin, as_index=False)[YCOL].mean().sort_values(x_bin)
    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=x_label, row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_quality_box(fig, df, row, col, color="#bdb76b"):
    d = df[[QCOL, YCOL, "Seq_Name"]].dropna().copy()

    # clean category strings
    d[QCOL] = d[QCOL].astype(str).str.strip()

    # Box
    fig.add_trace(
        go.Box(
            x=d[QCOL],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.50,
            showlegend=False,
        ),
        row=row, col=col
    )

    # Points
    fig.add_trace(
        go.Scatter(
            x=d[QCOL],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                "Quality casting: %{x}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text="Quality casting", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)

# -------------------------
# Layout: 2 rows × 3 cols (5 panels used)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Casting speed",
        "Mold width",
        "Quality casting",
        "SEN",
        "Argon flow",
        ""  # empty cell (row 2 col 3)
    ]
)

# Row 1
add_box_points_mean(fig, df, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]", row=1, col=1, bin_size=0.1)
add_box_points_mean(fig, df, "MOLD_WIDTH_avg [m]",        "Mold Width Avg [m]",        row=1, col=2, bin_size=0.05)
#add_quality_box(fig, df, row=1, col=3)

# Row 2
#add_scatter_mean(fig, df, "SEN_avg [mm]",          "SEN Avg [m]",            row=2, col=1, bin_size=0.001)  # 1 mm bins = 0.001 m
add_quality_box(fig, df, row=2, col=1)
add_scatter_mean(fig, df, "ArFlow_avg [NL/min]",   "Ar Flow Avg [NL/min]",   row=2, col=2, bin_size=0.5)
quality_order = (
    df.groupby("Quality casting")["MOLD_LEVEL_std [mm]"]
      .median()
      .sort_values()
      .index
      .tolist()
)

fig.update_xaxes(
    categoryorder="array",
    categoryarray=quality_order,
    row=1, col=3
)


# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1600,
    title="Strand #23-5 – Normal process only",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_5_recommended_correlations_NORMAL_only_plus_quality.html"
)

fig.show()


In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helpers
# -------------------------
def make_box_and_points(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace) for a given x_col vs Mold Level std.
    Boxes are grouped by rounded x to create meaningful distributions.
    Points are plotted at the same rounded x positions for a clean overlay.
    """
    x_round_name = f"{x_col}__rounded"
    df[x_round_name] = df[x_col].round(round_to)

    # Box trace (distribution per rounded bin)
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        boxmean="sd",
        marker=dict(color=color, opacity=0.85),
        line=dict(color=color, width=1),
        opacity=0.55,
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
    )

    # Scatter trace (points on top)
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.85),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False,
    )

    return box, scatter, x_round_name

# -------------------------
# Subplot layout (1x3)
# -------------------------
fig = make_subplots(
    rows=1, cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=[
        "mean DC Current, Low vs Mold Level σ (Normal only)",
        "mean DC Current, Up vs Mold Level σ (Normal only)",
        "mean AC Current, Up vs Mold Level σ (Normal only)",
    ],
)

panels = [
    (1, 1, "mean DC Current, Low [A]", "Mean DC Current, Low [A]", 0),  # round_to=0A
    (1, 2, "mean DC Current, Up [A]",  "Mean DC Current, Up [A]",  0),
    (1, 3, "mean AC Current, Up [A]",  "Mean AC Current, Up [A]",  0),
]

for r, c, xcol, xlabel, round_to in panels:
    box_tr, sc_tr, x_round_name = make_box_and_points(df, xcol, xlabel, round_to=round_to, color="#bdb76b")
    fig.add_trace(box_tr, row=r, col=c)
    fig.add_trace(sc_tr, row=r, col=c)

    fig.update_xaxes(title_text=f"{xlabel} (rounded)", row=r, col=c)

fig.update_yaxes(title_text="Mold Level σ [mm]", row=1, col=1)

fig.update_layout(
    height=520,
    width=1400,
    title="Strand #23-6 – Normal only: Mold Level σ vs mean currents (box + points)",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_currents_vs_ML_correlations_NORMAL_only_boxpoints.html"
)

fig.show()


In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helper: pure scatter
# -------------------------
def make_scatter(df, x_col, x_label, color="#bdb76b"):
    return go.Scatter(
        x=df[x_col],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(
            size=7,
            color=color,
            opacity=0.75,
        ),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label}: %{{x:.2f}}<br>"
            "Mold Level σ: %{y:.3f} mm"
            "<extra></extra>"
        ),
        showlegend=False,
    )

# -------------------------
# Subplots (1 × 3)
# -------------------------
fig = make_subplots(
    rows=1, cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=[
        "mean DC Current, Low vs Mold Level σ (Normal only)",
        "mean DC Current, Up vs Mold Level σ (Normal only)",
        "mean AC Current, Up vs Mold Level σ (Normal only)",
    ],
)

panels = [
    (1, 1, "mean DC Current, Low [A]", "Mean DC Current, Low [A]"),
    (1, 2, "mean DC Current, Up [A]",  "Mean DC Current, Up [A]"),
    (1, 3, "mean AC Current, Up [A]",  "Mean AC Current, Up [A]"),
]

for r, c, xcol, xlabel in panels:
    fig.add_trace(
        make_scatter(df, xcol, xlabel),
        row=r, col=c
    )
    fig.update_xaxes(title_text=xlabel, row=r, col=c)

fig.update_yaxes(title_text="Mold Level σ [mm]", row=1, col=1)

fig.update_layout(
    height=480,
    width=1400,
    title="Strand #23-6 – Normal process only: Mold Level σ vs mean currents",
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_6_currents_vs_ML_scatter_NORMAL_only.html"
)

fig.show()


In [0]:
import plotly.express as px

def plot_one_sequence(
    df_raw,
    df_seq,
    seq_idx,
    sequences,
    *,
    tcol="plainTimeStamp",
    value_col="Mold Level",
    title_prefix="",
):
    """
    Plot raw Mold Level time series for ONE sequence.

    Parameters
    ----------
    df_raw : pd.DataFrame
        Raw time-series dataframe (e.g. df_FCMold_ON_str23_6)
    df_seq : pd.DataFrame
        Aggregated per-sequence dataframe
        (output of generate_seq_statistics_with_disturbance)
    seq_idx : int
        Index of the sequence in df_seq / sequences
    sequences : list[list[int]]
        List of index groups returned by identify_sequences_segmented
    """

    # --- safety checks ---
    if seq_idx >= len(sequences):
        raise IndexError("seq_idx out of range")

    idxs = sequences[seq_idx]
    seg = df_raw.iloc[idxs].copy()

    # --- metadata ---
    meta = df_seq.iloc[seq_idx]

    title = (
        f"{title_prefix}"
        f"{meta['Seq_Name']} | "
        f"{meta['Seq_time_Start']} → {meta['Seq_time_End']}<br>"
        f"Vc={meta['CASTING_SPEED_avg [m/min]']:.2f}, "
        f"ML σ={meta['MOLD_LEVEL_std [mm]']:.2f} mm, "
        f"Disturbance={meta['has_disturbance']}"
    )

    # --- plot ---
    fig = px.line(
        seg,
        x=tcol,
        y=value_col,
        title=title,
        labels={
            tcol: "Time",
            value_col: "Mold Level [mm]",
        },
    )

    # reference lines
    fig.add_hline(
        y=seg[value_col].mean(),
        line_dash="dash",
        line_color="green",
        annotation_text="Mean",
        annotation_position="top left",
    )

    fig.add_hline(
        y=seg[value_col].min(),
        line_dash="dot",
        line_color="blue",
        annotation_text="Min",
        annotation_position="bottom left",
    )

    fig.add_hline(
        y=seg[value_col].max(),
        line_dash="dot",
        line_color="red",
        annotation_text="Max",
        annotation_position="top right",
    )

    fig.update_layout(
        height=400,
        width=1100,
        showlegend=False,
    )

    fig.show()


In [0]:
sequence = 127
plot_one_sequence(df_FCMold_ON_str23_6, df_seq, sequence, normal_groups_ON_str23_6)